# New Section

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]

        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

class TwoTowerDNN(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.alpha = 1.0
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)

        if hasattr(self, 'alpha') and self.alpha != 1.0:
            u_norm = torch.norm(u_rep, p=2, dim=1).clamp(min=1e-8)
            i_norm = torch.norm(i_rep, p=2, dim=1).clamp(min=1e-8)
            dot = dot / ((u_norm ** (1 - self.alpha)) * (i_norm ** (1 - self.alpha)))

        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# STEP 2: GRID SEARCH (10% SUB-SAMPLE)
# ---------------------------------------------------------
print("\n--- Starting Hyperparameter Grid Search (10% Data) ---")
sub_df = train_df.sample(frac=0.1, random_state=42)
u_arr_sub = sub_df['user_id'].values
i_arr_sub = sub_df['item_id'].values
r_arr_sub = sub_df['Rating'].values.astype(np.float32)

emb_dims = [32, 64]
hidden_dims = [32, 64]
lrs = [0.01, 0.005]

best_loss = float('inf')
best_params = {}

for ed in emb_dims:
    for hd in hidden_dims:
        for lr in lrs:
            print(f"Testing: Emb={ed}, Hidden={hd}, LR={lr}...")
            model_gs = TwoTowerDNN(num_users, num_items, emb_dim=ed, hidden_dim=hd).to(device)
            criterion_gs = nn.MSELoss()
            optimizer_gs = optim.Adam(model_gs.parameters(), lr=lr, weight_decay=1e-5)
            scaler_gs = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

            epochs_gs = 2
            batch_size_gs = 262144
            num_batches_gs = int(np.ceil(len(u_arr_sub) / batch_size_gs))

            model_gs.train()
            for epoch in range(epochs_gs):
                total_loss = 0.0
                indices = np.random.permutation(len(u_arr_sub))
                u_curr = u_arr_sub[indices]
                i_curr = i_arr_sub[indices]
                r_curr = r_arr_sub[indices]

                for b in range(num_batches_gs):
                    start_idx = b * batch_size_gs
                    end_idx = min((b + 1) * batch_size_gs, len(u_curr))

                    u = torch.from_numpy(u_curr[start_idx:end_idx]).to(device, non_blocking=True)
                    i = torch.from_numpy(i_curr[start_idx:end_idx]).to(device, non_blocking=True)
                    r = torch.from_numpy(r_curr[start_idx:end_idx]).to(device, non_blocking=True)

                    optimizer_gs.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                        preds = model_gs(u, i)
                        loss = criterion_gs(preds, r)

                    scaler_gs.scale(loss).backward()
                    scaler_gs.step(optimizer_gs)
                    scaler_gs.update()
                    total_loss += loss.item()

                final_epoch_loss = total_loss / num_batches_gs

            print(f"  -> Loss: {final_epoch_loss:.4f}")
            if final_epoch_loss < best_loss:
                best_loss = final_epoch_loss
                best_params = {'emb_dim': ed, 'hidden_dim': hd, 'lr': lr}

print("\n--- Grid Search Complete! ---")
print(f"Best Parameters Found: {best_params} (Loss: {best_loss:.4f})")

del sub_df, u_arr_sub, i_arr_sub, r_arr_sub, model_gs
gc.collect()

# ---------------------------------------------------------
# STEP 3: FULL OPTIMIZED TRAINING WITH BEST PARAMS
# ---------------------------------------------------------
print(f"\n--- Training Full Two-Tower DNN with Best Params ---")
epochs = 5
batch_size = 262144

model = TwoTowerDNN(num_users, num_items, emb_dim=best_params['emb_dim'], hidden_dim=best_params['hidden_dim']).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches = int(np.ceil(len(u_arr) / batch_size))

model.train()
start_time = time.time()

for epoch in range(epochs):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches):
        start_idx = b * batch_size
        end_idx = min((b + 1) * batch_size, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model(u, i)
            loss = criterion(preds, r)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | MSE Loss: {total_loss/num_batches:.4f}")

print(f"Training completed in {(time.time() - start_time)/60:.2f} minutes.")

print("Flushing training data from RAM to prepare for test evaluation...")
del u_arr, i_arr, r_arr, train_df
gc.collect()

torch.save(model.state_dict(), os.path.join(data_dir, "two_tower_model.pth"))
print("Model saved locally to 'two_tower_model.pth'.")

# ---------------------------------------------------------
# STEP 4: EVALUATION ON COMPLETE TEST DATA
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
    if 'Rating' not in test_df.columns and os.path.exists(probe_path):
        test_df = pd.read_parquet(probe_path)
elif os.path.exists(probe_path):
    test_df = pd.read_parquet(probe_path)
else:
    print("Error: Neither test.parquet nor probe_ratings.parquet found! Exiting.")
    exit(1)

if 'user_id' not in test_df.columns:
    test_df['user_id'] = test_df['CustomerID'].map(user_map)
if 'item_id' not in test_df.columns:
    test_df['item_id'] = test_df['MovieID'].map(movie_map)

print("\n--- Evaluating Rating Prediction Accuracy ---")
model.eval()

actuals = test_df['Rating'].values
global_average_rating = model.global_bias.item()
final_preds = np.full(len(test_df), global_average_rating)

valid_mask = test_df['user_id'].notna() & test_df['item_id'].notna()
valid_indices = np.where(valid_mask)[0]

if len(valid_indices) > 0:
    test_users = torch.tensor(test_df.loc[valid_mask, 'user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df.loc[valid_mask, 'item_id'].values, dtype=torch.long).to(device)

    with torch.no_grad():
        valid_preds = []
        batch_size = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)

rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
mae_score = np.mean(np.abs(actuals - final_preds))
print(f"Complete Test Records: {len(test_df):,}")
print(f"Complete Test RMSE: {rmse_score:.4f}")
print(f"Complete Test MAE:  {mae_score:.4f}")

print("\n--- Evaluating FULL-CORPUS MAP@10 (Using Dictionary Logic) ---")

valid_test_df = test_df[valid_mask]
all_test_users = valid_test_df['user_id'].unique()

# Sample 5,000 users randomly from the ENTIRE test set
import numpy as np
if len(all_test_users) > 5000:
    rng = np.random.default_rng(42)
    candidate_users = rng.choice(all_test_users, size=5000, replace=False)
else:
    candidate_users = all_test_users

print(f"Sampled {len(candidate_users):,} random users for MAP@10 Evaluation.")

# Build relevant_dict ONLY for the sampled users
relevant_test_df = valid_test_df[(valid_test_df['user_id'].isin(candidate_users)) & (valid_test_df['Rating'] >= 3.5)]
relevant_dict = (
    relevant_test_df.groupby('user_id')['item_id']
    .apply(set)
    .to_dict()
)

print("Building seen_dict for the sampled users...")
train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

# Grab target customer history NOW so we don't have to read it again in Step 5!
target_customer_id = 1488844
if target_customer_id not in user_map:
    target_customer_id = list(user_map.keys())[0]
target_user_idx = user_map[target_customer_id]

target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]

seen_dict = (
    train_history_filtered.groupby('user_id')['item_id']
    .apply(list)
    .to_dict()
)
del train_history_df, train_history_filtered
gc.collect()

print("Precomputing representations to accelerate scoring...")
model.eval()
all_items_tensor = torch.arange(num_items, dtype=torch.long, device=device)
with torch.no_grad():
    i_e = model.item_emb(all_items_tensor)
    i_rep = model.item_mlp(i_e)
    i_b = model.item_bias(all_items_tensor).squeeze()

    # Precompute all user reps
    all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
    u_e = model.user_emb(all_users_tensor)
    u_rep_all = model.user_mlp(u_e)
    u_b_all = model.user_bias(all_users_tensor).view(-1)

print(f"Vectorizing candidate scoring for {len(candidate_users):,} users...")
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

with torch.no_grad():
    # 1. Compute dense dot products: (5000, 32) @ (32, 17700) -> (5000, 17700)
    dense_dot = torch.matmul(u_rep_all, i_rep.T)

    # 2. Apply -inf mask to seen items so they can never be recommended
    for idx, u_id in enumerate(candidate_users):
        seen = seen_dict.get(u_id, [])
        if seen:
            dense_dot[idx, seen] = -float('inf')

    # 3. Precompute norms for alpha scaling
    u_norm_all = torch.norm(u_rep_all, p=2, dim=1).clamp(min=1e-8).unsqueeze(1) # (5000, 1)
    i_norm_all = torch.norm(i_rep, p=2, dim=1).clamp(min=1e-8).unsqueeze(0) # (1, 17700)

    best_map = 0.0
    best_alpha = 1.0

    print("\n--- Starting Grid Search for Optimal Alpha (Popularity Mitigation) ---", flush=True)
    for alpha in alphas:
        ap_list = []

        # Scale similarities dynamically
        sim = dense_dot / ((u_norm_all ** (1 - alpha)) * (i_norm_all ** (1 - alpha)))

        # Add biases
        scores = sim + u_b_all.unsqueeze(1) + i_b.unsqueeze(0) + model.global_bias

        # Find Top 10 for all users simultaneously
        top10_scores, top10_indices = torch.topk(scores, 10, dim=1)
        top10_items_all = top10_indices.cpu().numpy()

        # Calculate MAP
        for idx, u_id in enumerate(candidate_users):
            rel_set = relevant_dict.get(u_id, set())
            if not rel_set:
                ap_list.append(0.0)
                continue

            top10_items = top10_items_all[idx]

            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10_items, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

        map_at_10 = np.mean(ap_list) if ap_list else 0.0
        print(f"Alpha {alpha:.1f} -> MAP@10: {map_at_10:.4f}", flush=True)

        if map_at_10 > best_map:
            best_map = map_at_10
            best_alpha = alpha

print(f"\nOptimal Alpha found: {best_alpha:.1f} (MAP@10: {best_map:.4f})", flush=True)

# Assign best alpha to model so Step 5 uses it automatically
model.alpha = best_alpha

# Cleanup previous tensors
print("Cleaning up RAM...", flush=True)
if 'test_users' in locals():
    del test_users, test_items
del test_df, final_preds, valid_test_df, seen_dict, relevant_dict
gc.collect()

# ---------------------------------------------------------
# STEP 5: EXPLAINABLE TOP 10 RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"EXPLAINABLE TOP-10 RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s = set(target_user_history)
all_items = set(range(num_items))
cands = list(all_items - rated_s)

u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

print(f"Scoring {len(cands):,} candidate movies...")
with torch.no_grad():
    cands_preds_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_t.extend(batch_scores.cpu().numpy())

cands_preds = list(zip(cands, cands_preds_t))
cands_preds.sort(key=lambda x: x[1], reverse=True)
top_10 = cands_preds[:10]

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'DNN Score':<9} | {'Explanation (DNN CF Logic)'}")
print("-" * 130)
for rank, (item_idx, score) in enumerate(top_10, 1):
    movie_id = rev_movie_map[item_idx]
    title_str = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model, device)
    print(f"#{rank:<4} | {title_str:<50} | {score:<9.4f} | {explanation}")


Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Starting Hyperparameter Grid Search (10% Data) ---
Testing: Emb=32, Hidden=32, LR=0.01...
  -> Loss: 0.8546
Testing: Emb=32, Hidden=32, LR=0.005...
  -> Loss: 0.8548
Testing: Emb=32, Hidden=64, LR=0.01...
  -> Loss: 0.8513
Testing: Emb=32, Hidden=64, LR=0.005...
  -> Loss: 0.8479
Testing: Emb=64, Hidden=32, LR=0.01...
  -> Loss: 0.8596
Testing: Emb=64, Hidden=32, LR=0.005...
  -> Loss: 0.8561
Testing: Emb=64, Hidden=64, LR=0.01...
  -> Loss: 0.8550
Testing: Emb=64, Hidden=64, LR=0.005...
  -> Loss: 0.8502

--- Grid Search Complete! ---
Best Parameters Found: {'emb_dim': 32, 'hidden_dim': 64, 'lr': 0.005} (Loss: 0.8479)

--- Training Full Two-Tower DNN with Best Params ---
Epoch 1/5 | MSE Loss: 0.8295
Epoch 2/5 | MSE Loss: 0.7715
Epoch 3/5 | MSE Loss: 0.7500
Epoch 4/5 | MSE Loss: 0.7318
Epoch 5/5 | MSE Loss: 0.7238
Training compl

# New Section

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]

        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# Define Model 2: BPR Ranker
class TwoTowerBPR(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")
epochs_m1 = 5
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {total_loss/num_batches_m1:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - BPR w/ HARD NEGATIVES)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (BPR with Hard Negative Mining) ---")
epochs_m2 = 3
batch_size_m2 = 65536

model2 = TwoTowerBPR(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
optimizer2 = optim.Adam(model2.parameters(), lr=0.005, weight_decay=1e-5)
scaler2 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

num_batches_m2 = int(np.ceil(len(u_arr) / batch_size_m2))

print("Precomputing Model 1 representations to accelerate hard negative mining...")
model1.eval()
with torch.no_grad():
    # Precompute all 17,700 items (tiny matrix)
    all_items_t = torch.arange(num_items, device=device)
    all_i_rep1 = model1.item_mlp(model1.item_emb(all_items_t))
    all_i_b1 = model1.item_bias(all_items_t).squeeze(-1)

    # Precompute all 480k users (~60MB, very safe!)
    all_users_t = torch.arange(num_users, device=device)
    all_u_rep1 = model1.user_mlp(model1.user_emb(all_users_t))

model2.train()
start_time2 = time.time()

for epoch in range(epochs_m2):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]

    for b in range(num_batches_m2):
        start_idx = b * batch_size_m2
        end_idx = min((b + 1) * batch_size_m2, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer2.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            # Standard Random Negative Sampling (Restores Popularity MAP metrics)
            neg_i = torch.randint(0, num_items, (len(u),), device=device)

            # 2. BPR TRAINING FOR MODEL 2
            pos_preds = model2(u, i)
            neg_preds = model2(u, neg_i)
            bpr_loss = -F.logsigmoid(pos_preds - neg_preds).mean()

        scaler2.scale(bpr_loss).backward()
        scaler2.step(optimizer2)
        scaler2.update()

        total_loss += bpr_loss.item()

    print(f"Model 2 Epoch {epoch+1}/{epochs_m2} | BPR Loss: {total_loss/num_batches_m2:.4f}")

print(f"Model 2 Training completed in {(time.time() - start_time2)/60:.2f} minutes.")

print("Flushing training data from RAM to prepare for test evaluation...")
del u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
    print(f"Loaded test dataset: {len(test_df):,} ratings.")
else:
    print("WARNING: test.parquet not found. Skipping evaluation.")
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds = np.zeros(len(test_df), dtype=np.float32)
    valid_mask = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 MAP@10 ---")
    valid_test_df = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[(valid_test_df['user_id'].isin(candidate_users)) & (valid_test_df['Rating'] >= 3.5)]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model2.eval()
    all_items_tensor = torch.arange(num_items, dtype=torch.long, device=device)
    with torch.no_grad():
        i_e2 = model2.item_emb(all_items_tensor)
        i_rep2 = model2.item_mlp(i_e2)
        i_b2 = model2.item_bias(all_items_tensor).squeeze(-1)

        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e2 = model2.user_emb(all_users_tensor)
        u_rep_all2 = model2.user_mlp(u_e2)
        u_b2 = model2.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        # Standard MAP Evaluation (No penalty)
        dense_dot2 = torch.matmul(u_rep_all2, i_rep2.T)
        dense_scores = dense_dot2 + u_b2.unsqueeze(1) + i_b2.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores[idx, seen] = -float('inf')

        # Pure BPR ranking, no alpha scaling needed!
        top10_scores, top10_indices = torch.topk(dense_scores, 10, dim=1)
        top10_items_all = top10_indices.cpu().numpy()

        ap_list = []
        for idx, u_id in enumerate(candidate_users):
            rel_set = relevant_dict.get(u_id, set())
            if not rel_set:
                ap_list.append(0.0)
                continue

            top10_items = top10_items_all[idx]
            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10_items, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

        map_at_10 = np.mean(ap_list) if ap_list else 0.0
        print(f"Model 2 MAP@10: {map_at_10:.4f}")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s = set(target_user_history)
all_items = set(range(num_items))
cands = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_100_cands = [x[0] for x in cands_preds_m1[:100]]

# Stage 2: Re-Ranking using Model 2 with Bias Scaling
print(f"Stage 2: Re-Ranking candidates using Model 2 (BPR with 0.5 Popularity Penalty)...")
u_tensor_rank = torch.tensor([target_user_idx] * len(top_100_cands), dtype=torch.long).to(device)
i_tensor_rank = torch.tensor(top_100_cands, dtype=torch.long).to(device)

model2.eval()
with torch.no_grad():
    # Manually compute score to inject the popularity penalty
    u_rep = model2.user_mlp(model2.user_emb(u_tensor_rank))
    i_rep = model2.item_mlp(model2.item_emb(i_tensor_rank))
    dot = (u_rep * i_rep).sum(dim=1)

    u_b = model2.user_bias(u_tensor_rank).squeeze(-1)
    i_b = model2.item_bias(i_tensor_rank).squeeze(-1)

    # PENALTY: Multiply item_bias by 0.5 so it naturally balances Popularity vs Personalization
    cands_preds_m2_t = (dot + u_b + (i_b * 0.5)).cpu().numpy()

cands_preds_m2 = list(zip(top_100_cands, cands_preds_m2_t))
# Sort strictly by the penalized BPR score
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

# The Top 10 will now naturally be a mix of Blockbusters and Magic Recommendations!
hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"NATURAL HYBRID (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id = rev_movie_map[item_idx]
    title_str = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")



Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Model 1 Epoch 1/5 | MSE Loss: 0.8318
Model 1 Epoch 2/5 | MSE Loss: 0.7739
Model 1 Epoch 3/5 | MSE Loss: 0.7538
Model 1 Epoch 4/5 | MSE Loss: 0.7362
Model 1 Epoch 5/5 | MSE Loss: 0.7276
Model 1 Training completed in 1.70 minutes.

--- Training Stage 2: Re-Ranker (BPR with Hard Negative Mining) ---
Precomputing Model 1 representations to accelerate hard negative mining...
Model 2 Epoch 1/3 | BPR Loss: 0.2040
Model 2 Epoch 2/3 | BPR Loss: 0.1830
Model 2 Epoch 3/3 | BPR Loss: 0.1773
Model 2 Training completed in 2.23 minutes.
Flushing training data from RAM to prepare for test evaluation...

--- Loading Test Dataset into RAM ---
Loaded test dataset: 19,238,196 ratings.
Evaluating Model 1 (MSE) RMSE on complete test set...
Model 1 RMSE: 0.9850

--- Evaluating Model 2 MAP@10 ---
Model 2 

# New Section

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]

        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# Define Model 2: BPR Ranker
class TwoTowerBPR(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")
epochs_m1 = 5
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {total_loss/num_batches_m1:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - BPR w/ HARD NEGATIVES)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (BPR with Hard Negative Mining) ---")
epochs_m2 = 3
batch_size_m2 = 65536

model2 = TwoTowerBPR(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
optimizer2 = optim.Adam(model2.parameters(), lr=0.005, weight_decay=1e-5)
scaler2 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

num_batches_m2 = int(np.ceil(len(u_arr) / batch_size_m2))

print("Precomputing Model 1 representations to accelerate hard negative mining...")
model1.eval()
with torch.no_grad():
    # Precompute all 17,700 items (tiny matrix)
    all_items_t = torch.arange(num_items, device=device)
    all_i_rep1 = model1.item_mlp(model1.item_emb(all_items_t))
    all_i_b1 = model1.item_bias(all_items_t).squeeze(-1)

    # Precompute all 480k users (~60MB, very safe!)
    all_users_t = torch.arange(num_users, device=device)
    all_u_rep1 = model1.user_mlp(model1.user_emb(all_users_t))

model2.train()
start_time2 = time.time()

for epoch in range(epochs_m2):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]

    for b in range(num_batches_m2):
        start_idx = b * batch_size_m2
        end_idx = min((b + 1) * batch_size_m2, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer2.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            # Common positive predictions
            pos_preds = model2(u, i)

            # --- PURE RANDOM NEGATIVE SAMPLING ---
            rand_neg_i = torch.randint(0, num_items, (len(u),), device=device)
            rand_neg_preds = model2(u, rand_neg_i)
            bpr_loss = -F.logsigmoid(pos_preds - rand_neg_preds).mean()

        scaler2.scale(bpr_loss).backward()
        scaler2.step(optimizer2)
        scaler2.update()

        total_loss += bpr_loss.item()

    print(f"Model 2 Epoch {epoch+1}/{epochs_m2} | BPR Loss: {total_loss/num_batches_m2:.4f}")

print(f"Model 2 Training completed in {(time.time() - start_time2)/60:.2f} minutes.")

print("Flushing training data from RAM to prepare for test evaluation...")
del u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
    print(f"Loaded test dataset: {len(test_df):,} ratings.")
else:
    print("WARNING: test.parquet not found. Skipping evaluation.")
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds = np.zeros(len(test_df), dtype=np.float32)
    valid_mask = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 MAP@10 ---")
    valid_test_df = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[(valid_test_df['user_id'].isin(candidate_users)) & (valid_test_df['Rating'] >= 3.5)]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model2.eval()
    all_items_tensor = torch.arange(num_items, dtype=torch.long, device=device)
    with torch.no_grad():
        i_e2 = model2.item_emb(all_items_tensor)
        i_rep2 = model2.item_mlp(i_e2)
        i_b2 = model2.item_bias(all_items_tensor).squeeze(-1)

        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e2 = model2.user_emb(all_users_tensor)
        u_rep_all2 = model2.user_mlp(u_e2)
        u_b2 = model2.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        # Standard MAP Evaluation (No penalty)
        dense_dot2 = torch.matmul(u_rep_all2, i_rep2.T)
        dense_scores = dense_dot2 + u_b2.unsqueeze(1) + i_b2.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores[idx, seen] = -float('inf')

        # Pure BPR ranking, no alpha scaling needed!
        top10_scores, top10_indices = torch.topk(dense_scores, 10, dim=1)
        top10_items_all = top10_indices.cpu().numpy()

        ap_list = []
        for idx, u_id in enumerate(candidate_users):
            rel_set = relevant_dict.get(u_id, set())
            if not rel_set:
                ap_list.append(0.0)
                continue

            top10_items = top10_items_all[idx]
            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10_items, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

        map_at_10 = np.mean(ap_list) if ap_list else 0.0
        print(f"Model 2 MAP@10: {map_at_10:.4f}")

    # ---------------------------------------------------------
    # DIAGNOSTIC: Stage 1 Recall@100
    # How often does Model 1 even include the right movie in top 100?
    # ---------------------------------------------------------
    print("\n--- DIAGNOSTIC: Evaluating Model 1 Recall@100 ---")
    model1.eval()
    with torch.no_grad():
        i_e1 = model1.item_emb(all_items_tensor)
        i_rep1 = model1.item_mlp(i_e1)
        i_b1 = model1.item_bias(all_items_tensor).squeeze(-1)

        u_e1 = model1.user_emb(all_users_tensor)
        u_rep_all1 = model1.user_mlp(u_e1)
        u_b1 = model1.user_bias(all_users_tensor).squeeze(-1)

        dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores1[idx, seen] = -float('inf')

        # Get Top 100 for Model 1
        _, top100_indices1 = torch.topk(dense_scores1, 100, dim=1)
        top100_items_all1 = top100_indices1.cpu().numpy()

    hits = 0
    total = 0
    for idx, u_id in enumerate(candidate_users):
        rel_set = relevant_dict.get(u_id, set())
        if not rel_set:
            continue

        top100_items = set(top100_items_all1[idx])
        if any(item in top100_items for item in rel_set):
            hits += 1
        total += 1

    recall_at_100 = hits / total if total > 0 else 0
    print(f"Stage 1 Recall@100: {recall_at_100:.4f}")
    print(f"Interpreted: Model 1 includes at least 1 correct movie in {recall_at_100*100:.1f}% of users' Top 100")

    if recall_at_100 < 0.3:
        print("X CRITICAL: Model 1 is the bottleneck. Fix retrieval first.")
    elif recall_at_100 < 0.5:
        print("! WEAK: Model 1 is missing many correct movies.")
    else:
        print("OK: Model 1 retrieval is decent. Problem is in re-ranking.")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s = set(target_user_history)
all_items = set(range(num_items))
cands = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_100_cands = [x[0] for x in cands_preds_m1[:100]]

# Stage 2: Re-Ranking using Model 2 with Bias Scaling
print(f"Stage 2: Re-Ranking candidates using Model 2 (BPR with 0.5 Popularity Penalty)...")
u_tensor_rank = torch.tensor([target_user_idx] * len(top_100_cands), dtype=torch.long).to(device)
i_tensor_rank = torch.tensor(top_100_cands, dtype=torch.long).to(device)

model2.eval()
with torch.no_grad():
    # Manually compute score to inject the popularity penalty
    u_rep = model2.user_mlp(model2.user_emb(u_tensor_rank))
    i_rep = model2.item_mlp(model2.item_emb(i_tensor_rank))
    dot = (u_rep * i_rep).sum(dim=1)

    u_b = model2.user_bias(u_tensor_rank).squeeze(-1)
    i_b = model2.item_bias(i_tensor_rank).squeeze(-1)

    # PENALTY: Multiply item_bias by 0.5 so it naturally balances Popularity vs Personalization
    cands_preds_m2_t = (dot + u_b + (i_b * 0.5)).cpu().numpy()

cands_preds_m2 = list(zip(top_100_cands, cands_preds_m2_t))
# Sort strictly by the penalized BPR score
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

# The Top 10 will now naturally be a mix of Blockbusters and Magic Recommendations!
hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"NATURAL HYBRID (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id = rev_movie_map[item_idx]
    title_str = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")


Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Model 1 Epoch 1/5 | MSE Loss: 0.8327
Model 1 Epoch 2/5 | MSE Loss: 0.7687
Model 1 Epoch 3/5 | MSE Loss: 0.7494
Model 1 Epoch 4/5 | MSE Loss: 0.7348
Model 1 Epoch 5/5 | MSE Loss: 0.7278
Model 1 Training completed in 1.73 minutes.

--- Training Stage 2: Re-Ranker (BPR with Hard Negative Mining) ---
Precomputing Model 1 representations to accelerate hard negative mining...
Model 2 Epoch 1/3 | BPR Loss: 0.2030
Model 2 Epoch 2/3 | BPR Loss: 0.1814
Model 2 Epoch 3/3 | BPR Loss: 0.1769
Model 2 Training completed in 2.00 minutes.
Flushing training data from RAM to prepare for test evaluation...

--- Loading Test Dataset into RAM ---
Loaded test dataset: 19,238,196 ratings.
Evaluating Model 1 (MSE) RMSE on complete test set...
Model 1 RMSE: 0.9876

--- Evaluating Model 2 MAP@10 ---
Model 2 

# New Section

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]

        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")
epochs_m1 = 5
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {total_loss/num_batches_m1:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - LightGBM LambdaRank)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---")
import lightgbm as lgb

print("Extracting Last Watched movies for each user to compute Recency Score...")
train_dates_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Date'])
last_watched_df = train_dates_df.sort_values('Date').groupby('user_id').tail(1)
last_watched_map = dict(zip(last_watched_df['user_id'], last_watched_df['item_id']))
del train_dates_df
gc.collect()

print("Building LightGBM Training Dataset (Generating Model 1 Candidates)...")
train_users_unique = train_df['user_id'].unique()
rng = np.random.default_rng(42)
lgbm_train_users = rng.choice(train_users_unique, size=15000, replace=False)

lgbm_train_df = train_df[train_df['user_id'].isin(lgbm_train_users)].copy()
lgbm_train_df = lgbm_train_df[lgbm_train_df['Rating'] >= 4.0]
positive_interactions = lgbm_train_df.groupby('user_id')['item_id'].apply(set).to_dict()

model1.eval()
lgbm_features = []
lgbm_labels = []
lgbm_groups = []

all_items_t = torch.arange(num_items, device=device)
with torch.no_grad():
    i_e1 = model1.item_emb(all_items_t)
    i_rep1 = model1.item_mlp(i_e1)
    i_b1 = model1.item_bias(all_items_t).squeeze(-1)

for user in lgbm_train_users:
    pos_items = positive_interactions.get(user, set())
    if not pos_items:
        continue

    u_tensor = torch.tensor([user], device=device)
    with torch.no_grad():
        u_e1 = model1.user_emb(u_tensor)
        u_rep1 = model1.user_mlp(u_e1)
        u_b1 = model1.user_bias(u_tensor).squeeze(-1)

        dense_scores = torch.matmul(u_rep1, i_rep1.T).squeeze(0) + u_b1 + i_b1
        _, top100_idx = torch.topk(dense_scores, 100)
        top100 = top100_idx.cpu().numpy()

    last_item = last_watched_map.get(user, top100[0])
    last_item_tensor = torch.tensor([last_item], device=device)
    with torch.no_grad():
        last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

    cand_tensors = torch.tensor(top100, device=device)
    with torch.no_grad():
        cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))
        cand_norms = F.normalize(cand_reps1, p=2, dim=1)
        last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
        recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
        m1_scores = dense_scores[cand_tensors].cpu().numpy()

    for idx, cand in enumerate(top100):
        label = 1 if cand in pos_items else 0
        lgbm_features.append([cand, m1_scores[idx], recency_scores[idx]])
        lgbm_labels.append(label)

    lgbm_groups.append(len(top100))

X_train = pd.DataFrame(lgbm_features, columns=['movie_id', 'model1_score', 'recency_score'])
X_train['movie_id'] = X_train['movie_id'].astype('category')
y_train = np.array(lgbm_labels)

print("Training LightGBM Ranker with lambdarank objective...")
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
ranker.fit(
    X=X_train,
    y=y_train,
    group=lgbm_groups
)
print("LightGBM Training Completed.")

del lgbm_features, lgbm_labels, X_train, y_train, lgbm_train_df, u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
else:
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds = np.zeros(len(test_df), dtype=np.float32)
    valid_mask = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 (LightGBM) MAP@10 ---")
    valid_test_df = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[(valid_test_df['user_id'].isin(candidate_users)) & (valid_test_df['Rating'] >= 3.5)]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model1.eval()
    with torch.no_grad():
        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e1 = model1.user_emb(all_users_tensor)
        u_rep_all1 = model1.user_mlp(u_e1)
        u_b1 = model1.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores1[idx, seen] = -float('inf')

        _, top100_indices1 = torch.topk(dense_scores1, 100, dim=1)
        top100_items_all1 = top100_indices1.cpu().numpy()

    lgbm_test_features = []
    lgbm_test_indices = []

    for idx, u_id in enumerate(candidate_users):
        rel_set = relevant_dict.get(u_id, set())
        if not rel_set:
            continue

        top100_items = top100_items_all1[idx]
        last_item = last_watched_map.get(u_id, top100_items[0])

        last_item_tensor = torch.tensor([last_item], device=device)
        with torch.no_grad():
            last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

        cand_tensors = torch.tensor(top100_items, device=device)
        with torch.no_grad():
            cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))
            cand_norms = F.normalize(cand_reps1, p=2, dim=1)
            last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
            recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
            m1_scores = dense_scores1[idx, cand_tensors].cpu().numpy()

        for c_idx, cand in enumerate(top100_items):
            lgbm_test_features.append([cand, m1_scores[c_idx], recency_scores[c_idx]])
            lgbm_test_indices.append((idx, cand))

    ap_list = []
    if len(lgbm_test_features) > 0:
        X_test = pd.DataFrame(lgbm_test_features, columns=['movie_id', 'model1_score', 'recency_score'])
        X_test['movie_id'] = X_test['movie_id'].astype('category')
        lgbm_preds = ranker.predict(X_test)

        user_preds = {}
        for i, (u_idx, cand) in enumerate(lgbm_test_indices):
            if u_idx not in user_preds:
                user_preds[u_idx] = []
            user_preds[u_idx].append((cand, lgbm_preds[i]))

        for u_idx, preds in user_preds.items():
            u_id = candidate_users[u_idx]
            rel_set = relevant_dict.get(u_id, set())
            preds.sort(key=lambda x: x[1], reverse=True)
            top10 = [x[0] for x in preds[:10]]

            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

    map_at_10 = np.mean(ap_list) if ap_list else 0.0
    print(f"Model 2 (LightGBM) MAP@10: {map_at_10:.4f}")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s = set(target_user_history)
all_items = set(range(num_items))
cands = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_100_cands = [x[0] for x in cands_preds_m1[:100]]

# Stage 2: Re-Ranking using LightGBM
print(f"Stage 2: Re-Ranking candidates using LightGBM Ranker...")
target_last_item = last_watched_map.get(target_user_idx, top_100_cands[0])
target_last_item_tensor = torch.tensor([target_last_item], device=device)

with torch.no_grad():
    last_rep1 = model1.item_mlp(model1.item_emb(target_last_item_tensor)).squeeze(0)
    cand_tensors = torch.tensor(top_100_cands, device=device)
    cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))

    cand_norms = F.normalize(cand_reps1, p=2, dim=1)
    last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
    recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()

    # Ensure m1_scores perfectly align with the top_100_cands order
    m1_scores = np.array([x[1] for x in cands_preds_m1[:100]])

lgbm_target_features = []
for idx, cand in enumerate(top_100_cands):
    lgbm_target_features.append([cand, m1_scores[idx], recency_scores[idx]])

X_target = pd.DataFrame(lgbm_target_features, columns=['movie_id', 'model1_score', 'recency_score'])
X_target['movie_id'] = X_target['movie_id'].astype('category')
lgbm_target_preds = ranker.predict(X_target)

cands_preds_m2 = list(zip(top_100_cands, lgbm_target_preds))
# Sort strictly by the LightGBM predicted score
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

# The Top 10 are chosen by LightGBM's learned feature combination!
hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"LIGHTGBM RANKER (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id = rev_movie_map[item_idx]
    title_str = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")


Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Model 1 Epoch 1/5 | MSE Loss: 0.8310
Model 1 Epoch 2/5 | MSE Loss: 0.7755
Model 1 Epoch 3/5 | MSE Loss: 0.7520
Model 1 Epoch 4/5 | MSE Loss: 0.7360
Model 1 Epoch 5/5 | MSE Loss: 0.7281
Model 1 Training completed in 1.89 minutes.

--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---
Extracting Last Watched movies for each user to compute Recency Score...
Building LightGBM Training Dataset (Generating Model 1 Candidates)...
Training LightGBM Ranker with lambdarank objective...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of te

# New Section

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]

        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")
epochs_m1 = 5
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {total_loss/num_batches_m1:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - LightGBM LambdaRank)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---")
import lightgbm as lgb

print("Extracting Last Watched movies for each user to compute Recency Score...")
train_dates_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Date'])
last_watched_df = train_dates_df.sort_values('Date').groupby('user_id').tail(1)
last_watched_map = dict(zip(last_watched_df['user_id'], last_watched_df['item_id']))
del train_dates_df
gc.collect()

print("Calculating Item Popularity and Average Rating features...")
item_stats = train_df.groupby('item_id')['Rating'].agg(['count', 'mean']).reset_index()
item_pop_map = dict(zip(item_stats['item_id'], item_stats['count']))
item_avg_map = dict(zip(item_stats['item_id'], item_stats['mean']))

print("Building LightGBM Training Dataset (Generating Model 1 Candidates)...")
train_users_unique = train_df['user_id'].unique()
rng = np.random.default_rng(42)
lgbm_train_users = rng.choice(train_users_unique, size=15000, replace=False)

lgbm_train_df = train_df[train_df['user_id'].isin(lgbm_train_users)].copy()
lgbm_train_df = lgbm_train_df[lgbm_train_df['Rating'] >= 4.0]
positive_interactions = lgbm_train_df.groupby('user_id')['item_id'].apply(set).to_dict()

model1.eval()
lgbm_features = []
lgbm_labels = []
lgbm_groups = []

all_items_t = torch.arange(num_items, device=device)
with torch.no_grad():
    i_e1 = model1.item_emb(all_items_t)
    i_rep1 = model1.item_mlp(i_e1)
    i_b1 = model1.item_bias(all_items_t).squeeze(-1)

for user in lgbm_train_users:
    pos_items = positive_interactions.get(user, set())
    if not pos_items:
        continue

    u_tensor = torch.tensor([user], device=device)
    with torch.no_grad():
        u_e1 = model1.user_emb(u_tensor)
        u_rep1 = model1.user_mlp(u_e1)
        u_b1 = model1.user_bias(u_tensor).squeeze(-1)

        dense_scores = torch.matmul(u_rep1, i_rep1.T).squeeze(0) + u_b1 + i_b1
        _, top100_idx = torch.topk(dense_scores, 100)
        top100 = top100_idx.cpu().numpy()

    last_item = last_watched_map.get(user, top100[0])
    last_item_tensor = torch.tensor([last_item], device=device)
    with torch.no_grad():
        last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

    cand_tensors = torch.tensor(top100, device=device)
    with torch.no_grad():
        cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))
        cand_norms = F.normalize(cand_reps1, p=2, dim=1)
        last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
        recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
        m1_scores = dense_scores[cand_tensors].cpu().numpy()

    for idx, cand in enumerate(top100):
        label = 1 if cand in pos_items else 0
        pop = item_pop_map.get(cand, 0)
        avg_r = item_avg_map.get(cand, 3.5)
        lgbm_features.append([m1_scores[idx], recency_scores[idx], pop, avg_r])
        lgbm_labels.append(label)

    lgbm_groups.append(len(top100))

X_train = pd.DataFrame(lgbm_features, columns=['model1_score', 'recency_score', 'item_popularity', 'item_avg_rating'])
y_train = np.array(lgbm_labels)

print("Training LightGBM Ranker with lambdarank objective...")
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
ranker.fit(
    X=X_train,
    y=y_train,
    group=lgbm_groups
)
print("LightGBM Training Completed.")

del lgbm_features, lgbm_labels, X_train, y_train, lgbm_train_df, u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
else:
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds = np.zeros(len(test_df), dtype=np.float32)
    valid_mask = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 (LightGBM) MAP@10 ---")
    valid_test_df = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[(valid_test_df['user_id'].isin(candidate_users)) & (valid_test_df['Rating'] >= 3.5)]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model1.eval()
    with torch.no_grad():
        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e1 = model1.user_emb(all_users_tensor)
        u_rep_all1 = model1.user_mlp(u_e1)
        u_b1 = model1.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores1[idx, seen] = -float('inf')

        _, top100_indices1 = torch.topk(dense_scores1, 100, dim=1)
        top100_items_all1 = top100_indices1.cpu().numpy()

    lgbm_test_features = []
    lgbm_test_indices = []

    for idx, u_id in enumerate(candidate_users):
        rel_set = relevant_dict.get(u_id, set())
        if not rel_set:
            continue

        top100_items = top100_items_all1[idx]
        last_item = last_watched_map.get(u_id, top100_items[0])

        last_item_tensor = torch.tensor([last_item], device=device)
        with torch.no_grad():
            last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

        cand_tensors = torch.tensor(top100_items, device=device)
        with torch.no_grad():
            cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))
            cand_norms = F.normalize(cand_reps1, p=2, dim=1)
            last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
            recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
            m1_scores = dense_scores1[idx, cand_tensors].cpu().numpy()

        for c_idx, cand in enumerate(top100_items):
            pop = item_pop_map.get(cand, 0)
            avg_r = item_avg_map.get(cand, 3.5)
            lgbm_test_features.append([m1_scores[c_idx], recency_scores[c_idx], pop, avg_r])
            lgbm_test_indices.append((idx, cand))

    ap_list = []
    if len(lgbm_test_features) > 0:
        X_test = pd.DataFrame(lgbm_test_features, columns=['model1_score', 'recency_score', 'item_popularity', 'item_avg_rating'])
        lgbm_preds = ranker.predict(X_test)

        user_preds = {}
        for i, (u_idx, cand) in enumerate(lgbm_test_indices):
            if u_idx not in user_preds:
                user_preds[u_idx] = []
            user_preds[u_idx].append((cand, lgbm_preds[i]))

        for u_idx, preds in user_preds.items():
            u_id = candidate_users[u_idx]
            rel_set = relevant_dict.get(u_id, set())
            preds.sort(key=lambda x: x[1], reverse=True)
            top10 = [x[0] for x in preds[:10]]

            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

    map_at_10 = np.mean(ap_list) if ap_list else 0.0
    print(f"Model 2 (LightGBM) MAP@10: {map_at_10:.4f}")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s = set(target_user_history)
all_items = set(range(num_items))
cands = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_100_cands = [x[0] for x in cands_preds_m1[:100]]

# Stage 2: Re-Ranking using LightGBM
print(f"Stage 2: Re-Ranking candidates using LightGBM Ranker...")
target_last_item = last_watched_map.get(target_user_idx, top_100_cands[0])
target_last_item_tensor = torch.tensor([target_last_item], device=device)

with torch.no_grad():
    last_rep1 = model1.item_mlp(model1.item_emb(target_last_item_tensor)).squeeze(0)
    cand_tensors = torch.tensor(top_100_cands, device=device)
    cand_reps1 = model1.item_mlp(model1.item_emb(cand_tensors))

    cand_norms = F.normalize(cand_reps1, p=2, dim=1)
    last_norm = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
    recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()

    # Ensure m1_scores perfectly align with the top_100_cands order
    m1_scores = np.array([x[1] for x in cands_preds_m1[:100]])

lgbm_target_features = []
for idx, cand in enumerate(top_100_cands):
    pop = item_pop_map.get(cand, 0)
    avg_r = item_avg_map.get(cand, 3.5)
    lgbm_target_features.append([m1_scores[idx], recency_scores[idx], pop, avg_r])

X_target = pd.DataFrame(lgbm_target_features, columns=['model1_score', 'recency_score', 'item_popularity', 'item_avg_rating'])
lgbm_target_preds = ranker.predict(X_target)

cands_preds_m2 = list(zip(top_100_cands, lgbm_target_preds))
# Sort strictly by the LightGBM predicted score
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

# The Top 10 are chosen by LightGBM's learned feature combination!
hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"LIGHTGBM RANKER (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id = rev_movie_map[item_idx]
    title_str = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")

Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Model 1 Epoch 1/5 | MSE Loss: 0.8333
Model 1 Epoch 2/5 | MSE Loss: 0.7758
Model 1 Epoch 3/5 | MSE Loss: 0.7570
Model 1 Epoch 4/5 | MSE Loss: 0.7406
Model 1 Epoch 5/5 | MSE Loss: 0.7295
Model 1 Training completed in 1.94 minutes.

--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---
Extracting Last Watched movies for each user to compute Recency Score...
Calculating Item Popularity and Average Rating features...
Building LightGBM Training Dataset (Generating Model 1 Candidates)...
Training LightGBM Ranker with lambdarank objective...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003295 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Tota

# New Section

In [ ]:
import os
import gc
import time
import math
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]
        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
movie_years = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"
                if m_id.isdigit() and int(m_id) in movie_map:
                    try:
                        movie_years[movie_map[int(m_id)]] = int(year)
                    except ValueError:
                        movie_years[movie_map[int(m_id)]] = 2000

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")

def evaluate_rmse(model, u_tensor, i_tensor, r_tensor, batch_size=262144):
    model.eval()
    total_loss = 0.0
    num_batches = int(np.ceil(len(u_tensor) / batch_size))
    criterion = nn.MSELoss()
    with torch.no_grad():
        for b in range(num_batches):
            start_idx = b * batch_size
            end_idx = min((b + 1) * batch_size, len(u_tensor))
            u = u_tensor[start_idx:end_idx]
            i = i_tensor[start_idx:end_idx]
            r = r_tensor[start_idx:end_idx]
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                preds = model(u, i)
                loss = criterion(preds, r)
            total_loss += loss.item() * len(u)
    model.train()
    return np.sqrt(total_loss / len(u_tensor))

print("Extracting 1M evaluation samples for Model 1...")
rng = np.random.default_rng(42)

train_sample_idx = rng.choice(len(train_df), size=1000000, replace=False)
train_sample_df = train_df.iloc[train_sample_idx]
eval_train_u = torch.tensor(train_sample_df['user_id'].values, dtype=torch.long).to(device)
eval_train_i = torch.tensor(train_sample_df['item_id'].values, dtype=torch.long).to(device)
eval_train_r = torch.tensor(train_sample_df['Rating'].values, dtype=torch.float32).to(device)
del train_sample_df

if os.path.exists(test_path):
    test_eval_df = pd.read_parquet(test_path)
    test_eval_df = test_eval_df[test_eval_df['user_id'].isin(user_map.values()) & test_eval_df['item_id'].isin(movie_map.values())]
    if len(test_eval_df) > 1000000:
        test_sample_idx = rng.choice(len(test_eval_df), size=1000000, replace=False)
        test_eval_df = test_eval_df.iloc[test_sample_idx]

    eval_test_u = torch.tensor(test_eval_df['user_id'].values, dtype=torch.long).to(device)
    eval_test_i = torch.tensor(test_eval_df['item_id'].values, dtype=torch.long).to(device)
    eval_test_r = torch.tensor(test_eval_df['Rating'].values, dtype=torch.float32).to(device)
    del test_eval_df
else:
    eval_test_u, eval_test_i, eval_test_r = None, None, None

epochs_m1 = 10
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    avg_loss = total_loss / num_batches_m1
    train_rmse = evaluate_rmse(model1, eval_train_u, eval_train_i, eval_train_r) if eval_train_u is not None else 0.0
    test_rmse = evaluate_rmse(model1, eval_test_u, eval_test_i, eval_test_r) if eval_test_u is not None else 0.0

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {avg_loss:.4f} | Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - LightGBM LambdaRank)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---")
import lightgbm as lgb

print("Extracting Last Watched movies for each user to compute Recency Score...")
train_dates_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Date'])
last_watched_df = train_dates_df.sort_values('Date').groupby('user_id').tail(1)
last_watched_map = dict(zip(last_watched_df['user_id'], last_watched_df['item_id']))
del train_dates_df
gc.collect()

# -------------------------------------------------------
# CHANGE 1: Extended item stats — added std for rating_std feature
# -------------------------------------------------------
print("Calculating Item Popularity, Average Rating, and Std features...")
item_stats = train_df.groupby('item_id')['Rating'].agg(['count', 'mean', 'std']).reset_index()
item_pop_map  = dict(zip(item_stats['item_id'], item_stats['count']))
item_avg_map  = dict(zip(item_stats['item_id'], item_stats['mean']))
item_std_map  = dict(zip(item_stats['item_id'], item_stats['std'].fillna(0)))

# -------------------------------------------------------
# CHANGE 1 (cont): User average rating — for rating_deviation feature
# -------------------------------------------------------
print("Calculating User Average Rating feature...")
user_avg_map = dict(train_df.groupby('user_id')['Rating'].mean())

# -------------------------------------------------------
# CHANGE 2: Increased training users from 15,000 → 30,000
# -------------------------------------------------------
print("Building LightGBM Training Dataset (Generating Model 1 Candidates)...")
train_users_unique = train_df['user_id'].unique()
rng = np.random.default_rng(42)
lgbm_train_users = rng.choice(train_users_unique, size=15000, replace=False)  # was 30000

# FIX: use Rating >= 3.5 to match evaluation threshold (was 4.0 — misaligned)
lgbm_train_df = train_df[train_df['user_id'].isin(lgbm_train_users)].copy()
lgbm_train_df = lgbm_train_df[lgbm_train_df['Rating'] >= 3.5]
positive_interactions = lgbm_train_df.groupby('user_id')['item_id'].apply(set).to_dict()

model1.eval()
max_train_rows = len(lgbm_train_users) * 1000
lgbm_features = np.zeros((max_train_rows, 74), dtype=np.float32)
lgbm_labels = np.zeros(max_train_rows, dtype=np.int8)
lgbm_groups = []
row_idx = 0

all_items_t = torch.arange(num_items, device=device)
with torch.no_grad():
    i_e1   = model1.item_emb(all_items_t)
    i_rep1 = model1.item_mlp(i_e1)
    i_b1   = model1.item_bias(all_items_t).squeeze(-1)

for user in lgbm_train_users:
    pos_items = positive_interactions.get(user, set())
    if not pos_items:
        continue

    u_tensor = torch.tensor([user], device=device)
    with torch.no_grad():
        u_e1  = model1.user_emb(u_tensor)
        u_rep1 = model1.user_mlp(u_e1)
        u_b1  = model1.user_bias(u_tensor).squeeze(-1)

        dense_scores = torch.matmul(u_rep1, i_rep1.T).squeeze(0) + u_b1 + i_b1
        # expanded from 100 → 200: wider pool = more ground truth items captured
        _, top1000_idx = torch.topk(dense_scores, 1000)
        top1000 = top1000_idx.cpu().numpy()

    last_item = last_watched_map.get(user, top1000[0])
    last_item_tensor = torch.tensor([last_item], device=device)
    with torch.no_grad():
        last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

    cand_tensors = torch.tensor(top1000, device=device)
    with torch.no_grad():
        cand_reps1    = model1.item_mlp(model1.item_emb(cand_tensors))
        cand_norms    = F.normalize(cand_reps1, p=2, dim=1)
        last_norm     = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
        recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
        m1_scores     = dense_scores[cand_tensors].cpu().numpy()

    u_vec = u_rep1.cpu().numpy()[0]
    cand_vecs = cand_reps1.cpu().numpy()

    user_avg = user_avg_map.get(user, 3.5)

    pops = [item_pop_map.get(c, 0) for c in top1000]
    max_pop = max(pops) if pops and max(pops) > 0 else 1

    for idx, cand in enumerate(top1000):
        label      = 1 if cand in pos_items else 0
        pop        = item_pop_map.get(cand, 0)
        avg_r      = item_avg_map.get(cand, 3.5)
        rating_std = item_std_map.get(cand, 0)
        # rank position: 1-based, normalised by pool size so it's comparable across users
        m1_rank    = (idx + 1) / 1000.0

        year       = movie_years.get(cand, 2000)
        rating_gap = avg_r - user_avg
        rel_pop    = pop / max_pop

        lgbm_features[row_idx, :10] = [
            m1_scores[idx],
            recency_scores[idx],
            pop,
            avg_r,
            rating_std,
            user_avg,
            m1_rank,
            year,
            rating_gap,
            rel_pop
        ]
        lgbm_features[row_idx, 10:42] = u_vec
        lgbm_features[row_idx, 42:74] = cand_vecs[idx]

        lgbm_labels[row_idx] = label
        row_idx += 1

    lgbm_groups.append(len(top1000))

lgbm_features = lgbm_features[:row_idx]
lgbm_labels = lgbm_labels[:row_idx]

FEATURE_COLS = [
    'model1_score',
    'recency_score',
    'item_popularity',
    'item_avg_rating',
    'rating_std',
    'user_avg_rating',
    'model1_rank',
    'item_release_year',
    'rating_gap',
    'relative_popularity'
] + [f'u_emb_{i}' for i in range(32)] + [f'i_emb_{i}' for i in range(32)]

X_train = pd.DataFrame(lgbm_features, columns=FEATURE_COLS)
y_train = np.array(lgbm_labels)

print("Training LightGBM Ranker with lambdarank objective...")
num_groups  = len(lgbm_groups)
split_idx   = int(num_groups * 0.8)

train_groups = lgbm_groups[:split_idx]
val_groups   = lgbm_groups[split_idx:]

row_split_idx  = sum(train_groups)
X_train_split  = X_train.iloc[:row_split_idx]
y_train_split  = y_train[:row_split_idx]
X_val_split    = X_train.iloc[row_split_idx:]
y_val_split    = y_train[row_split_idx:]

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=200,       # was 100 — valid NDCG was still climbing
    learning_rate=0.05,     # slower + more careful with more trees
    num_leaves=63,          # was default 31 — more expressive trees
    min_child_samples=20,   # avoid overfitting on rare items
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)
ranker.fit(
    X=X_train_split,
    y=y_train_split,
    group=train_groups,
    eval_set=[(X_train_split, y_train_split), (X_val_split, y_val_split)],
    eval_group=[train_groups, val_groups],
    eval_names=['train', 'valid'],
    eval_at=[10],
    callbacks=[lgb.log_evaluation(period=1)]
)
print("LightGBM Training Completed.")

del lgbm_features, lgbm_labels, X_train, y_train, lgbm_train_df, u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
else:
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals    = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds  = np.zeros(len(test_df), dtype=np.float32)
    valid_mask   = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size  = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 (LightGBM) MAP@10 ---")
    valid_test_df  = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[
        (valid_test_df['user_id'].isin(candidate_users)) &
        (valid_test_df['Rating'] >= 3.5)
    ]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx     = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model1.eval()
    with torch.no_grad():
        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e1         = model1.user_emb(all_users_tensor)
        u_rep_all1   = model1.user_mlp(u_e1)
        u_b1         = model1.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores1[idx, seen] = -float('inf')

        _, top100_indices1   = torch.topk(dense_scores1, 1000, dim=1)  # expanded 100 → 200
        top100_items_all1    = top100_indices1.cpu().numpy()

    max_test_rows = len(candidate_users) * 1000
    lgbm_test_features = np.zeros((max_test_rows, 74), dtype=np.float32)
    lgbm_test_indices  = []
    test_row_idx = 0

    for idx, u_id in enumerate(candidate_users):
        rel_set = relevant_dict.get(u_id, set())
        if not rel_set:
            continue

        top100_items   = top100_items_all1[idx]   # now 200 candidates
        last_item      = last_watched_map.get(u_id, top100_items[0])

        last_item_tensor = torch.tensor([last_item], device=device)
        with torch.no_grad():
            last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

        cand_tensors = torch.tensor(top100_items, device=device)
        with torch.no_grad():
            cand_reps1     = model1.item_mlp(model1.item_emb(cand_tensors))
            cand_norms     = F.normalize(cand_reps1, p=2, dim=1)
            last_norm      = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
            recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
            m1_scores      = dense_scores1[idx, cand_tensors].cpu().numpy()

        u_vec = u_rep_all1[idx].cpu().numpy()
        cand_vecs = cand_reps1.cpu().numpy()

        user_avg = user_avg_map.get(u_id, 3.5)

        pops = [item_pop_map.get(c, 0) for c in top100_items]
        max_pop = max(pops) if pops and max(pops) > 0 else 1

        for c_idx, cand in enumerate(top100_items):
            pop        = item_pop_map.get(cand, 0)
            avg_r      = item_avg_map.get(cand, 3.5)
            rating_std = item_std_map.get(cand, 0)
            m1_rank    = (c_idx + 1) / 1000.0

            year       = movie_years.get(cand, 2000)
            rating_gap = avg_r - user_avg
            rel_pop    = pop / max_pop

            lgbm_test_features[test_row_idx, :10] = [
                m1_scores[c_idx],
                recency_scores[c_idx],
                pop,
                avg_r,
                rating_std,
                user_avg,
                m1_rank,
                year,
                rating_gap,
                rel_pop
            ]
            lgbm_test_features[test_row_idx, 10:42] = u_vec
            lgbm_test_features[test_row_idx, 42:74] = cand_vecs[c_idx]

            lgbm_test_indices.append((idx, cand))
            test_row_idx += 1

    lgbm_test_features = lgbm_test_features[:test_row_idx]

    ap_list = []
    if len(lgbm_test_features) > 0:
        X_test      = pd.DataFrame(lgbm_test_features, columns=FEATURE_COLS)
        lgbm_preds  = ranker.predict(X_test)

        user_preds = {}
        for i, (u_idx, cand) in enumerate(lgbm_test_indices):
            if u_idx not in user_preds:
                user_preds[u_idx] = []
            user_preds[u_idx].append((cand, lgbm_preds[i]))

        for u_idx, preds in user_preds.items():
            u_id    = candidate_users[u_idx]
            rel_set = relevant_dict.get(u_id, set())
            preds.sort(key=lambda x: x[1], reverse=True)
            top10   = [x[0] for x in preds[:10]]

            num_hits       = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

    map_at_10 = np.mean(ap_list) if ap_list else 0.0
    print(f"Model 2 (LightGBM) MAP@10: {map_at_10:.4f}")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s   = set(target_user_history)
all_items = set(range(num_items))
cands     = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_1000_cands = [x[0] for x in cands_preds_m1[:1000]]  # expanded 100 → 200

# Stage 2: Re-Ranking using LightGBM
print(f"Stage 2: Re-Ranking candidates using LightGBM Ranker...")
target_last_item        = last_watched_map.get(target_user_idx, top_1000_cands[0])
target_last_item_tensor = torch.tensor([target_last_item], device=device)

with torch.no_grad():
    last_rep1    = model1.item_mlp(model1.item_emb(target_last_item_tensor)).squeeze(0)
    cand_tensors = torch.tensor(top_1000_cands, device=device)
    cand_reps1   = model1.item_mlp(model1.item_emb(cand_tensors))

    cand_norms     = F.normalize(cand_reps1, p=2, dim=1)
    last_norm      = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
    recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
    m1_scores      = np.array([x[1] for x in cands_preds_m1[:1000]])

    u_tensor = torch.tensor([target_user_idx], device=device)
    u_rep1   = model1.user_mlp(model1.user_emb(u_tensor))

u_vec = u_rep1.cpu().numpy()[0]
cand_vecs = cand_reps1.cpu().numpy()

target_user_avg    = user_avg_map.get(target_user_idx, 3.5)

pops = [item_pop_map.get(c, 0) for c in top_1000_cands]
max_pop = max(pops) if pops and max(pops) > 0 else 1

lgbm_target_features = np.zeros((len(top_1000_cands), 74), dtype=np.float32)
for idx, cand in enumerate(top_1000_cands):
    pop        = item_pop_map.get(cand, 0)
    avg_r      = item_avg_map.get(cand, 3.5)
    rating_std = item_std_map.get(cand, 0)
    m1_rank    = (idx + 1) / 1000.0

    year       = movie_years.get(cand, 2000)
    rating_gap = avg_r - target_user_avg
    rel_pop    = pop / max_pop

    lgbm_target_features[idx, :10] = [
        m1_scores[idx],
        recency_scores[idx],
        pop,
        avg_r,
        rating_std,
        target_user_avg,
        m1_rank,
        year,
        rating_gap,
        rel_pop
    ]
    lgbm_target_features[idx, 10:42] = u_vec
    lgbm_target_features[idx, 42:74] = cand_vecs[idx]

X_target         = pd.DataFrame(lgbm_target_features, columns=FEATURE_COLS)
lgbm_target_preds = ranker.predict(X_target)

cands_preds_m2 = list(zip(top_1000_cands, lgbm_target_preds))
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"LIGHTGBM RANKER (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id    = rev_movie_map[item_idx]
    title_str   = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")


Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Extracting 1M evaluation samples for Model 1...
Model 1 Epoch 1/10 | MSE Loss: 0.8308 | Train RMSE: 0.8803 | Test RMSE: 0.9936
Model 1 Epoch 2/10 | MSE Loss: 0.7725 | Train RMSE: 0.8623 | Test RMSE: 0.9889
Model 1 Epoch 3/10 | MSE Loss: 0.7519 | Train RMSE: 0.8519 | Test RMSE: 0.9881
Model 1 Epoch 4/10 | MSE Loss: 0.7363 | Train RMSE: 0.8431 | Test RMSE: 0.9858
Model 1 Epoch 5/10 | MSE Loss: 0.7285 | Train RMSE: 0.8395 | Test RMSE: 0.9849
Model 1 Epoch 6/10 | MSE Loss: 0.7233 | Train RMSE: 0.8355 | Test RMSE: 0.9842
Model 1 Epoch 7/10 | MSE Loss: 0.7182 | Train RMSE: 0.8322 | Test RMSE: 0.9862
Model 1 Epoch 8/10 | MSE Loss: 0.7147 | Train RMSE: 0.8296 | Test RMSE: 0.9843
Model 1 Epoch 9/10 | MSE Loss: 0.7120 | Train RMSE: 0.8286 | Test RMSE: 0.9857
Model 1 Epoch 10/10 | MSE Loss: 0

# New Section

In [ ]:
import os
import gc
import time
import math
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."

train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
probe_path = os.path.join(data_dir, "probe_ratings.parquet")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: CUDA GPU not found. Training on CPU will be significantly slower!")

def compute_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

def get_cf_user_explanation_dnn(rec_item_idx, target_user_idx, train_path, model, device):
    try:
        with torch.no_grad():
            u_tensor = torch.tensor([target_user_idx], dtype=torch.long).to(device)
            target_vector = model.user_mlp(model.user_emb(u_tensor)).cpu().numpy()[0]
    except Exception:
        return "Based on deep latent profile fit."

    try:
        item_ratings = pd.read_parquet(
            train_path,
            columns=['user_id', 'Rating', 'CustomerID'],
            filters=[('item_id', '==', rec_item_idx), ('Rating', '>=', 4)]
        )
    except Exception:
        return "Based on deep latent profile fit."

    if item_ratings.empty:
        return "Based on deep latent profile fit."

    peer_uids = item_ratings['user_id'].values
    valid_peers = [uid for uid in peer_uids if uid != target_user_idx]

    if not valid_peers:
        return "Based on deep latent profile fit."

    with torch.no_grad():
        peer_tensors = torch.tensor(valid_peers, dtype=torch.long).to(device)
        peer_vectors = model.user_mlp(model.user_emb(peer_tensors)).cpu().numpy()

    similarities = []
    for i, uid in enumerate(valid_peers):
        cust_id = item_ratings[item_ratings['user_id'] == uid]['CustomerID'].values[0]
        r = item_ratings[item_ratings['user_id'] == uid]['Rating'].values[0]
        peer_vector = peer_vectors[i]
        sim = compute_cosine_similarity(target_vector, peer_vector)
        similarities.append((cust_id, r, sim))

    if not similarities:
        return "Based on deep latent profile fit."

    similarities.sort(key=lambda x: x[2], reverse=True)
    top_peers = similarities[:2]

    peer_exps = [f"Customer {cust_id} ({rating}★, Similarity: {sim:.2f})" for cust_id, rating, sim in top_peers]
    return f"Users with similar tastes (like {', '.join(peer_exps)}) rated this movie highly."

# ---------------------------------------------------------
# STEP 1: LOAD ONLY TRAINING DATA
# ---------------------------------------------------------
print("\n--- Loading Training Data & Mappings ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
rev_movie_map = {idx: mid for mid, idx in movie_map.items()}

movie_titles = {}
movie_years = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line):
            f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                movie_titles[int(m_id)] = f"{title.strip()} ({year})"
                if m_id.isdigit() and int(m_id) in movie_map:
                    try:
                        movie_years[movie_map[int(m_id)]] = int(year)
                    except ValueError:
                        movie_years[movie_map[int(m_id)]] = 2000

num_users = len(user_map)
num_items = len(movie_map)

print("Loading train.parquet into RAM...")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating'])
print(f"Loaded the complete training dataset: {len(train_df):,} ratings...")

# Define Model 1: MSE Candidate Generator
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# STEP 2: TRAIN MODEL 1 (CANDIDATE GENERATOR - MSE)
# ---------------------------------------------------------
print("\n--- Training Stage 1: Candidate Generator (MSE) ---")

def evaluate_rmse(model, u_tensor, i_tensor, r_tensor, batch_size=262144):
    model.eval()
    total_loss = 0.0
    num_batches = int(np.ceil(len(u_tensor) / batch_size))
    criterion = nn.MSELoss()
    with torch.no_grad():
        for b in range(num_batches):
            start_idx = b * batch_size
            end_idx = min((b + 1) * batch_size, len(u_tensor))
            u = u_tensor[start_idx:end_idx]
            i = i_tensor[start_idx:end_idx]
            r = r_tensor[start_idx:end_idx]
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                preds = model(u, i)
                loss = criterion(preds, r)
            total_loss += loss.item() * len(u)
    model.train()
    return np.sqrt(total_loss / len(u_tensor))

print("Extracting 1M evaluation samples for Model 1...")
rng = np.random.default_rng(42)

train_sample_idx = rng.choice(len(train_df), size=1000000, replace=False)
train_sample_df = train_df.iloc[train_sample_idx]
eval_train_u = torch.tensor(train_sample_df['user_id'].values, dtype=torch.long).to(device)
eval_train_i = torch.tensor(train_sample_df['item_id'].values, dtype=torch.long).to(device)
eval_train_r = torch.tensor(train_sample_df['Rating'].values, dtype=torch.float32).to(device)
del train_sample_df

if os.path.exists(test_path):
    test_eval_df = pd.read_parquet(test_path)
    test_eval_df = test_eval_df[test_eval_df['user_id'].isin(user_map.values()) & test_eval_df['item_id'].isin(movie_map.values())]
    if len(test_eval_df) > 1000000:
        test_sample_idx = rng.choice(len(test_eval_df), size=1000000, replace=False)
        test_eval_df = test_eval_df.iloc[test_sample_idx]

    eval_test_u = torch.tensor(test_eval_df['user_id'].values, dtype=torch.long).to(device)
    eval_test_i = torch.tensor(test_eval_df['item_id'].values, dtype=torch.long).to(device)
    eval_test_r = torch.tensor(test_eval_df['Rating'].values, dtype=torch.float32).to(device)
    del test_eval_df
else:
    eval_test_u, eval_test_i, eval_test_r = None, None, None

epochs_m1 = 10
batch_size_m1 = 262144

model1 = TwoTowerMSE(num_users, num_items, emb_dim=64, hidden_dim=32).to(device)
criterion1 = nn.MSELoss()
optimizer1 = optim.Adam(model1.parameters(), lr=0.005, weight_decay=1e-5)
scaler1 = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

u_arr = train_df['user_id'].values
i_arr = train_df['item_id'].values
r_arr = train_df['Rating'].values.astype(np.float32)
num_batches_m1 = int(np.ceil(len(u_arr) / batch_size_m1))

model1.train()
start_time1 = time.time()

for epoch in range(epochs_m1):
    total_loss = 0.0
    indices = np.random.permutation(len(u_arr))
    u_arr = u_arr[indices]
    i_arr = i_arr[indices]
    r_arr = r_arr[indices]

    for b in range(num_batches_m1):
        start_idx = b * batch_size_m1
        end_idx = min((b + 1) * batch_size_m1, len(u_arr))

        u = torch.from_numpy(u_arr[start_idx:end_idx]).to(device, non_blocking=True)
        i = torch.from_numpy(i_arr[start_idx:end_idx]).to(device, non_blocking=True)
        r = torch.from_numpy(r_arr[start_idx:end_idx]).to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            preds = model1(u, i)
            loss = criterion1(preds, r)

        scaler1.scale(loss).backward()
        scaler1.step(optimizer1)
        scaler1.update()

        total_loss += loss.item()

    avg_loss = total_loss / num_batches_m1
    train_rmse = evaluate_rmse(model1, eval_train_u, eval_train_i, eval_train_r) if eval_train_u is not None else 0.0
    test_rmse = evaluate_rmse(model1, eval_test_u, eval_test_i, eval_test_r) if eval_test_u is not None else 0.0

    print(f"Model 1 Epoch {epoch+1}/{epochs_m1} | MSE Loss: {avg_loss:.4f} | Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

print(f"Model 1 Training completed in {(time.time() - start_time1)/60:.2f} minutes.")

print("\n--- Saving Model 1 ---")
torch.save(model1, 'model1_retrieval_2000.pkl')
print("Saved model1_retrieval_2000.pkl")

# ---------------------------------------------------------
# STEP 3: TRAIN MODEL 2 (RANKER - LightGBM LambdaRank)
# ---------------------------------------------------------
print("\n--- Training Stage 2: Re-Ranker (LightGBM LambdaRank) ---")
import lightgbm as lgb

print("Extracting Last Watched movies for each user to compute Recency Score...")
train_dates_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Date'])
last_watched_df = train_dates_df.sort_values('Date').groupby('user_id').tail(1)
last_watched_map = dict(zip(last_watched_df['user_id'], last_watched_df['item_id']))
del train_dates_df
gc.collect()

# -------------------------------------------------------
# CHANGE 1: Extended item stats — added std for rating_std feature
# -------------------------------------------------------
print("Calculating Item Popularity, Average Rating, and Std features...")
item_stats = train_df.groupby('item_id')['Rating'].agg(['count', 'mean', 'std']).reset_index()
item_pop_map  = dict(zip(item_stats['item_id'], item_stats['count']))
item_avg_map  = dict(zip(item_stats['item_id'], item_stats['mean']))
item_std_map  = dict(zip(item_stats['item_id'], item_stats['std'].fillna(0)))

# -------------------------------------------------------
# CHANGE 1 (cont): User average rating — for rating_deviation feature
# -------------------------------------------------------
print("Calculating User Average Rating feature...")
user_avg_map = dict(train_df.groupby('user_id')['Rating'].mean())

# -------------------------------------------------------
# CHANGE 2: Increased training users from 15,000 → 30,000
# -------------------------------------------------------
print("Building LightGBM Training Dataset (Generating Model 1 Candidates)...")
train_users_unique = train_df['user_id'].unique()
rng = np.random.default_rng(42)
lgbm_train_users = rng.choice(train_users_unique, size=15000, replace=False)  # was 30000

# FIX: use Rating >= 3.5 to match evaluation threshold (was 4.0 — misaligned)
lgbm_train_df = train_df[train_df['user_id'].isin(lgbm_train_users)].copy()
lgbm_train_df = lgbm_train_df[lgbm_train_df['Rating'] >= 3.5]
positive_interactions = lgbm_train_df.groupby('user_id')['item_id'].apply(set).to_dict()

model1.eval()
max_train_rows = len(lgbm_train_users) * 2000
lgbm_features = np.zeros((max_train_rows, 74), dtype=np.float16)
lgbm_labels = np.zeros(max_train_rows, dtype=np.int8)
lgbm_groups = []
row_idx = 0

all_items_t = torch.arange(num_items, device=device)
with torch.no_grad():
    i_e1   = model1.item_emb(all_items_t)
    i_rep1 = model1.item_mlp(i_e1)
    i_b1   = model1.item_bias(all_items_t).squeeze(-1)

for user in lgbm_train_users:
    pos_items = positive_interactions.get(user, set())
    if not pos_items:
        continue

    u_tensor = torch.tensor([user], device=device)
    with torch.no_grad():
        u_e1  = model1.user_emb(u_tensor)
        u_rep1 = model1.user_mlp(u_e1)
        u_b1  = model1.user_bias(u_tensor).squeeze(-1)

        dense_scores = torch.matmul(u_rep1, i_rep1.T).squeeze(0) + u_b1 + i_b1
        # expanded from 100 → 200: wider pool = more ground truth items captured
        _, top2000_idx = torch.topk(dense_scores, 2000)
        top2000 = top2000_idx.cpu().numpy()

    last_item = last_watched_map.get(user, top2000[0])
    last_item_tensor = torch.tensor([last_item], device=device)
    with torch.no_grad():
        last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

    cand_tensors = torch.tensor(top2000, device=device)
    with torch.no_grad():
        cand_reps1    = model1.item_mlp(model1.item_emb(cand_tensors))
        cand_norms    = F.normalize(cand_reps1, p=2, dim=1)
        last_norm     = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
        recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
        m1_scores     = dense_scores[cand_tensors].cpu().numpy()

    u_vec = u_rep1.cpu().numpy()[0]
    cand_vecs = cand_reps1.cpu().numpy()

    user_avg = user_avg_map.get(user, 3.5)

    pops = [item_pop_map.get(c, 0) for c in top2000]
    max_pop = max(pops) if pops and max(pops) > 0 else 1

    for idx, cand in enumerate(top2000):
        label      = 1 if cand in pos_items else 0
        pop        = item_pop_map.get(cand, 0)
        avg_r      = item_avg_map.get(cand, 3.5)
        rating_std = item_std_map.get(cand, 0)
        # rank position: 1-based, normalised by pool size so it's comparable across users
        m1_rank    = (idx + 1) / 2000.0

        year       = movie_years.get(cand, 2000)
        rating_gap = avg_r - user_avg
        rel_pop    = pop / max_pop

        lgbm_features[row_idx, :10] = [
            m1_scores[idx],
            recency_scores[idx],
            np.log1p(pop),
            avg_r,
            rating_std,
            user_avg,
            m1_rank,
            year,
            rating_gap,
            rel_pop
        ]
        lgbm_features[row_idx, 10:42] = u_vec
        lgbm_features[row_idx, 42:74] = cand_vecs[idx]

        lgbm_labels[row_idx] = label
        row_idx += 1

    lgbm_groups.append(len(top2000))

lgbm_features = lgbm_features[:row_idx]
lgbm_labels = lgbm_labels[:row_idx]

FEATURE_COLS = [
    'model1_score',
    'recency_score',
    'item_popularity',
    'item_avg_rating',
    'rating_std',
    'user_avg_rating',
    'model1_rank',
    'item_release_year',
    'rating_gap',
    'relative_popularity'
] + [f'u_emb_{i}' for i in range(32)] + [f'i_emb_{i}' for i in range(32)]

print("Splitting numpy arrays for LightGBM...")
num_groups  = len(lgbm_groups)
split_idx   = int(num_groups * 0.8)

train_groups = lgbm_groups[:split_idx]
val_groups   = lgbm_groups[split_idx:]

row_split_idx  = sum(train_groups)
X_train_split  = lgbm_features[:row_split_idx]
y_train_split  = lgbm_labels[:row_split_idx]
X_val_split    = lgbm_features[row_split_idx:]
y_val_split    = lgbm_labels[row_split_idx:]

del lgbm_features, lgbm_labels
gc.collect()

print("Training LightGBM Ranker with lambdarank objective...")

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=200,       # was 100 — valid NDCG was still climbing
    learning_rate=0.05,     # slower + more careful with more trees
    num_leaves=63,          # was default 31 — more expressive trees
    min_child_samples=20,   # avoid overfitting on rare items
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)
ranker.fit(
    X=X_train_split,
    y=y_train_split,
    group=train_groups,
    eval_set=[(X_train_split, y_train_split), (X_val_split, y_val_split)],
    eval_group=[train_groups, val_groups],
    eval_names=['train', 'valid'],
    eval_at=[10],
    feature_name=FEATURE_COLS,
    callbacks=[lgb.log_evaluation(period=1)]
)

print("\n--- Saving Model 2 (LightGBM) ---")
import joblib
joblib.dump(ranker, 'ranker_lightgbm_2000.pkl')
print("Saved ranker_lightgbm_2000.pkl")
print("LightGBM Training Completed.")

del X_train_split, y_train_split, X_val_split, y_val_split, lgbm_train_df, u_arr, i_arr, r_arr, train_df
gc.collect()

# ---------------------------------------------------------
# STEP 4: EVALUATION
# ---------------------------------------------------------
print("\n--- Loading Test Dataset into RAM ---")
if os.path.exists(test_path):
    test_df = pd.read_parquet(test_path)
else:
    test_df = pd.DataFrame()

if not test_df.empty:
    test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]
    test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
    test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
    actuals    = test_df['Rating'].values

    print("Evaluating Model 1 (MSE) RMSE on complete test set...")
    model1.eval()
    final_preds  = np.zeros(len(test_df), dtype=np.float32)
    valid_mask   = actuals > 0
    valid_indices = np.where(valid_mask)[0]

    with torch.no_grad():
        valid_preds = []
        batch_size  = 10000
        for idx in range(0, len(test_users), batch_size):
            batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
            valid_preds.extend(batch_preds.cpu().numpy())

    final_preds[valid_indices] = np.array(valid_preds)
    rmse_score = np.sqrt(np.mean((actuals - final_preds) ** 2))
    print(f"Model 1 RMSE: {rmse_score:.4f}")

    print("\n--- Evaluating Model 2 (LightGBM) MAP@10 ---")
    valid_test_df  = test_df[valid_mask]
    all_test_users = valid_test_df['user_id'].unique()

    if len(all_test_users) > 5000:
        rng = np.random.default_rng(42)
        candidate_users = rng.choice(all_test_users, size=5000, replace=False)
    else:
        candidate_users = all_test_users

    relevant_test_df = valid_test_df[
        (valid_test_df['user_id'].isin(candidate_users)) &
        (valid_test_df['Rating'] >= 3.5)
    ]
    relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

    train_history_df = pd.read_parquet(train_path, columns=['user_id', 'item_id'])

    target_customer_id = 1488844
    if target_customer_id not in user_map:
        target_customer_id = list(user_map.keys())[0]
    target_user_idx     = user_map[target_customer_id]
    target_user_history = train_history_df[train_history_df['user_id'] == target_user_idx]['item_id'].values

    train_history_filtered = train_history_df[train_history_df['user_id'].isin(candidate_users)]
    seen_dict = train_history_filtered.groupby('user_id')['item_id'].apply(list).to_dict()
    del train_history_df, train_history_filtered
    gc.collect()

    model1.eval()
    with torch.no_grad():
        all_users_tensor = torch.tensor(candidate_users, dtype=torch.long, device=device)
        u_e1         = model1.user_emb(all_users_tensor)
        u_rep_all1   = model1.user_mlp(u_e1)
        u_b1         = model1.user_bias(all_users_tensor).squeeze(-1)

    with torch.no_grad():
        dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

        for idx, u_id in enumerate(candidate_users):
            seen = seen_dict.get(u_id, [])
            if seen:
                dense_scores1[idx, seen] = -float('inf')

        _, top100_indices1   = torch.topk(dense_scores1, 2000, dim=1)  # expanded 100 → 200
        top100_items_all1    = top100_indices1.cpu().numpy()

    max_test_rows = len(candidate_users) * 2000
    lgbm_test_features = np.zeros((max_test_rows, 74), dtype=np.float16)
    lgbm_test_indices  = []
    test_row_idx = 0

    for idx, u_id in enumerate(candidate_users):
        rel_set = relevant_dict.get(u_id, set())
        if not rel_set:
            continue

        top100_items   = top100_items_all1[idx]   # now 200 candidates
        last_item      = last_watched_map.get(u_id, top100_items[0])

        last_item_tensor = torch.tensor([last_item], device=device)
        with torch.no_grad():
            last_rep1 = model1.item_mlp(model1.item_emb(last_item_tensor)).squeeze(0)

        cand_tensors = torch.tensor(top100_items, device=device)
        with torch.no_grad():
            cand_reps1     = model1.item_mlp(model1.item_emb(cand_tensors))
            cand_norms     = F.normalize(cand_reps1, p=2, dim=1)
            last_norm      = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
            recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
            m1_scores      = dense_scores1[idx, cand_tensors].cpu().numpy()

        u_vec = u_rep_all1[idx].cpu().numpy()
        cand_vecs = cand_reps1.cpu().numpy()

        user_avg = user_avg_map.get(u_id, 3.5)

        pops = [item_pop_map.get(c, 0) for c in top100_items]
        max_pop = max(pops) if pops and max(pops) > 0 else 1

        for c_idx, cand in enumerate(top100_items):
            pop        = item_pop_map.get(cand, 0)
            avg_r      = item_avg_map.get(cand, 3.5)
            rating_std = item_std_map.get(cand, 0)
            m1_rank    = (c_idx + 1) / 2000.0

            year       = movie_years.get(cand, 2000)
            rating_gap = avg_r - user_avg
            rel_pop    = pop / max_pop

            lgbm_test_features[test_row_idx, :10] = [
                m1_scores[c_idx],
                recency_scores[c_idx],
                np.log1p(pop),
                avg_r,
                rating_std,
                user_avg,
                m1_rank,
                year,
                rating_gap,
                rel_pop
            ]
            lgbm_test_features[test_row_idx, 10:42] = u_vec
            lgbm_test_features[test_row_idx, 42:74] = cand_vecs[c_idx]

            lgbm_test_indices.append((idx, cand))
            test_row_idx += 1

    lgbm_test_features = lgbm_test_features[:test_row_idx]

    ap_list = []
    if len(lgbm_test_features) > 0:
        lgbm_preds  = ranker.predict(lgbm_test_features)

        user_preds = {}
        for i, (u_idx, cand) in enumerate(lgbm_test_indices):
            if u_idx not in user_preds:
                user_preds[u_idx] = []
            user_preds[u_idx].append((cand, lgbm_preds[i]))

        for u_idx, preds in user_preds.items():
            u_id    = candidate_users[u_idx]
            rel_set = relevant_dict.get(u_id, set())
            preds.sort(key=lambda x: x[1], reverse=True)
            top10   = [x[0] for x in preds[:10]]

            num_hits       = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

    map_at_10 = np.mean(ap_list) if ap_list else 0.0
    print(f"Model 2 (LightGBM) MAP@10: {map_at_10:.4f}")

# ---------------------------------------------------------
# STEP 5: TWO-STAGE HYBRID RECOMMENDATIONS
# ---------------------------------------------------------
print("\n" + "="*80)
print(f"TWO-STAGE HYBRID RECOMMENDATIONS FOR CUSTOMER {target_customer_id}")
print("="*80)

rated_s   = set(target_user_history)
all_items = set(range(num_items))
cands     = list(all_items - rated_s)

# Stage 1: Candidate Generation (Top 100) using Model 1
print(f"Stage 1: Retrieving Top 100 Candidates using Model 1 (MSE)...")
u_tensor = torch.tensor([target_user_idx] * len(cands), dtype=torch.long).to(device)
i_tensor = torch.tensor(cands, dtype=torch.long).to(device)

model1.eval()
with torch.no_grad():
    cands_preds_m1_t = []
    batch_size = 10000
    for idx in range(0, len(u_tensor), batch_size):
        batch_scores = model1(u_tensor[idx:idx+batch_size], i_tensor[idx:idx+batch_size])
        cands_preds_m1_t.extend(batch_scores.cpu().numpy())

cands_preds_m1 = list(zip(cands, cands_preds_m1_t))
cands_preds_m1.sort(key=lambda x: x[1], reverse=True)
top_2000_cands = [x[0] for x in cands_preds_m1[:2000]]  # expanded 100 → 200

# Stage 2: Re-Ranking using LightGBM
print(f"Stage 2: Re-Ranking candidates using LightGBM Ranker...")
target_last_item        = last_watched_map.get(target_user_idx, top_2000_cands[0])
target_last_item_tensor = torch.tensor([target_last_item], device=device)

with torch.no_grad():
    last_rep1    = model1.item_mlp(model1.item_emb(target_last_item_tensor)).squeeze(0)
    cand_tensors = torch.tensor(top_2000_cands, device=device)
    cand_reps1   = model1.item_mlp(model1.item_emb(cand_tensors))

    cand_norms     = F.normalize(cand_reps1, p=2, dim=1)
    last_norm      = F.normalize(last_rep1.unsqueeze(0), p=2, dim=1)
    recency_scores = (cand_norms * last_norm).sum(dim=1).cpu().numpy()
    m1_scores      = np.array([x[1] for x in cands_preds_m1[:2000]])

    u_tensor = torch.tensor([target_user_idx], device=device)
    u_rep1   = model1.user_mlp(model1.user_emb(u_tensor))

u_vec = u_rep1.cpu().numpy()[0]
cand_vecs = cand_reps1.cpu().numpy()

target_user_avg    = user_avg_map.get(target_user_idx, 3.5)

pops = [item_pop_map.get(c, 0) for c in top_2000_cands]
max_pop = max(pops) if pops and max(pops) > 0 else 1

lgbm_target_features = np.zeros((len(top_2000_cands), 74), dtype=np.float16)
for idx, cand in enumerate(top_2000_cands):
    pop        = item_pop_map.get(cand, 0)
    avg_r      = item_avg_map.get(cand, 3.5)
    rating_std = item_std_map.get(cand, 0)
    m1_rank    = (idx + 1) / 2000.0

    year       = movie_years.get(cand, 2000)
    rating_gap = avg_r - target_user_avg
    rel_pop    = pop / max_pop

    lgbm_target_features[idx, :10] = [
        m1_scores[idx],
        recency_scores[idx],
        np.log1p(pop),
        avg_r,
        rating_std,
        target_user_avg,
        m1_rank,
        year,
        rating_gap,
        rel_pop
    ]
    lgbm_target_features[idx, 10:42] = u_vec
    lgbm_target_features[idx, 42:74] = cand_vecs[idx]

lgbm_target_preds = ranker.predict(lgbm_target_features)

cands_preds_m2 = list(zip(top_2000_cands, lgbm_target_preds))
cands_preds_m2.sort(key=lambda x: x[1], reverse=True)

hybrid_top_10 = []
for item_idx, score in cands_preds_m2[:10]:
    hybrid_top_10.append((item_idx, f"LIGHTGBM RANKER (Score: {score:.4f})"))

print(f"\n{'Rank':<5} | {'Movie Title (Year)':<50} | {'Recommendation Type':<30} | {'Explanation (DNN CF Logic)'}")
print("-" * 140)
for rank, (item_idx, rec_type) in enumerate(hybrid_top_10, 1):
    movie_id    = rev_movie_map[item_idx]
    title_str   = movie_titles.get(movie_id, f"Movie {movie_id}")
    explanation = get_cf_user_explanation_dnn(item_idx, target_user_idx, train_path, model1, device)
    print(f"#{rank:<4} | {title_str:<50} | {rec_type:<30} | {explanation}")


Using device: cuda

--- Loading Training Data & Mappings ---
Loading train.parquet into RAM...
Loaded the complete training dataset: 77,135,487 ratings...

--- Training Stage 1: Candidate Generator (MSE) ---
Extracting 1M evaluation samples for Model 1...
Model 1 Epoch 1/10 | MSE Loss: 0.8301 | Train RMSE: 0.8813 | Test RMSE: 0.9947
Model 1 Epoch 2/10 | MSE Loss: 0.7739 | Train RMSE: 0.8629 | Test RMSE: 0.9884
Model 1 Epoch 3/10 | MSE Loss: 0.7511 | Train RMSE: 0.8497 | Test RMSE: 0.9883
Model 1 Epoch 4/10 | MSE Loss: 0.7351 | Train RMSE: 0.8425 | Test RMSE: 0.9856
Model 1 Epoch 5/10 | MSE Loss: 0.7272 | Train RMSE: 0.8379 | Test RMSE: 0.9858
Model 1 Epoch 6/10 | MSE Loss: 0.7222 | Train RMSE: 0.8355 | Test RMSE: 0.9865
Model 1 Epoch 7/10 | MSE Loss: 0.7189 | Train RMSE: 0.8332 | Test RMSE: 0.9850
Model 1 Epoch 8/10 | MSE Loss: 0.7157 | Train RMSE: 0.8305 | Test RMSE: 0.9841
Model 1 Epoch 9/10 | MSE Loss: 0.7135 | Train RMSE: 0.8296 | Test RMSE: 0.9840
Model 1 Epoch 10/10 | MSE Loss: 0

# New Section

In [ ]:
import os
import gc
import math
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

# ---------------------------------------------------------
# CONFIGURATION & PATHS
# ---------------------------------------------------------
data_dir = "."
train_path = os.path.join(data_dir, "train.parquet")
test_path = os.path.join(data_dir, "test.parquet")
maps_path = os.path.join(data_dir, "maps.pkl")
titles_file = os.path.join(data_dir, "movie_titles.csv")
model1_path = os.path.join(data_dir, "model1_retrieval_2000.pkl")
model2_path = os.path.join(data_dir, "ranker_lightgbm_2000.pkl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------------------------------------
# DEFINE MODEL 1 CLASS (Required for PyTorch unpickling)
# ---------------------------------------------------------
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# 1. LOAD MAPS AND MODELS
# ---------------------------------------------------------
print("\n--- Loading Data and Models ---")
with open(maps_path, "rb") as f:
    maps = pickle.load(f)
user_map = maps['user_map']
movie_map = maps['movie_map']
num_items = len(movie_map)

print(f"Loading Model 1 from {model1_path}...")
model1 = torch.load(model1_path, map_location=device, weights_only=False)
model1.eval()

print(f"Loading Model 2 from {model2_path}...")
ranker = joblib.load(model2_path)

print("Loading metadata...")
movie_years = {}
if os.path.exists(titles_file):
    with open(titles_file, "r", encoding="ISO-8859-1") as f:
        first_line = f.readline()
        if not ("MovieID" in first_line or "movie_id" in first_line): f.seek(0)
        for line in f:
            parts = line.strip().split(",", 2)
            if len(parts) == 3:
                m_id, year, title = parts
                if m_id.isdigit() and int(m_id) in movie_map:
                    try:
                        movie_years[movie_map[int(m_id)]] = int(year)
                    except ValueError:
                        movie_years[movie_map[int(m_id)]] = 2000

# ---------------------------------------------------------
# 2. CALCULATE COLD/WARM STATS FROM TRAIN SET
# ---------------------------------------------------------
print("\n--- Calculating Cold/Warm definitions from Training Set ---")
train_df = pd.read_parquet(train_path, columns=['user_id', 'item_id', 'Rating', 'Date'])

user_counts = train_df['user_id'].value_counts()
movie_counts = train_df['item_id'].value_counts()

cold_users_set = set(user_counts[user_counts <= 50].index)
warm_users_set = set(user_counts[user_counts > 50].index)

cold_movies_set = set(movie_counts[movie_counts < 100].index)
warm_movies_set = set(movie_counts[movie_counts >= 100].index)

print(f"Total Cold Users (<= 50 ratings): {len(cold_users_set):,}")
print(f"Total Warm Users (> 50 ratings):  {len(warm_users_set):,}")

# Precompute features required for LightGBM
print("Precomputing statistics for LightGBM features...")
try:
    train_df['Date'] = pd.to_datetime(train_df['Date'])
    idx = train_df.groupby('user_id')['Date'].idxmax()
    last_watched_df = train_df.loc[idx]
except Exception:
    # Fallback if Date parsing fails (still uses much less memory than sort_values)
    last_watched_df = train_df.drop_duplicates('user_id', keep='last')

last_watched_map = dict(zip(last_watched_df['user_id'], last_watched_df['item_id']))
del last_watched_df
gc.collect()

item_stats = train_df.groupby('item_id')['Rating'].agg(['count', 'mean', 'std']).reset_index()
item_pop_map  = dict(zip(item_stats['item_id'], item_stats['count']))
item_avg_map  = dict(zip(item_stats['item_id'], item_stats['mean']))
item_std_map  = dict(zip(item_stats['item_id'], item_stats['std'].fillna(0)))

user_avg_map = dict(train_df.groupby('user_id')['Rating'].mean())

# Extract raw arrays to avoid holding 100M row Pandas DataFrames in memory
train_u_array = train_df['user_id'].values.copy()
train_i_array = train_df['item_id'].values.copy()

del train_df
gc.collect()

# ---------------------------------------------------------
# 3. RMSE EVALUATION (Cold vs Warm slices)
# ---------------------------------------------------------
print("\n--- Evaluating Model 1 (MSE) RMSE ---")
test_df = pd.read_parquet(test_path)
test_df = test_df[test_df['user_id'].isin(user_map.values()) & test_df['item_id'].isin(movie_map.values())]

test_users = torch.tensor(test_df['user_id'].values, dtype=torch.long).to(device)
test_items = torch.tensor(test_df['item_id'].values, dtype=torch.long).to(device)
actuals    = test_df['Rating'].values

final_preds  = np.zeros(len(test_df), dtype=np.float32)
valid_mask   = actuals > 0
valid_indices = np.where(valid_mask)[0]

with torch.no_grad():
    valid_preds = []
    batch_size  = 10000
    for idx in range(0, len(test_users), batch_size):
        batch_preds = model1(test_users[idx:idx+batch_size], test_items[idx:idx+batch_size])
        valid_preds.extend(batch_preds.cpu().numpy())

final_preds[valid_indices] = np.array(valid_preds)
test_df['Pred'] = final_preds

valid_test_df = test_df[valid_mask].copy()
valid_test_df['err_sq'] = (valid_test_df['Rating'] - valid_test_df['Pred']) ** 2

# Cold vs Warm Masks
cold_u_mask = valid_test_df['user_id'].isin(cold_users_set)
warm_u_mask = valid_test_df['user_id'].isin(warm_users_set)

print(f"Overall RMSE:     {np.sqrt(valid_test_df['err_sq'].mean()):.4f}")
print(f"Cold Users RMSE:  {np.sqrt(valid_test_df[cold_u_mask]['err_sq'].mean()):.4f} (Count: {cold_u_mask.sum():,})")
print(f"Warm Users RMSE:  {np.sqrt(valid_test_df[warm_u_mask]['err_sq'].mean()):.4f} (Count: {warm_u_mask.sum():,})")

# ---------------------------------------------------------
# 4. MAP@10 EVALUATION (Cold vs Warm Users)
# ---------------------------------------------------------
print("\n--- Evaluating Model 2 (LightGBM) MAP@10 ---")
FEATURE_COLS = [
    'model1_score', 'recency_score', 'item_popularity', 'item_avg_rating',
    'rating_std', 'user_avg_rating', 'model1_rank', 'item_release_year',
    'rating_gap', 'relative_popularity'
] + [f'u_emb_{i}' for i in range(32)] + [f'i_emb_{i}' for i in range(32)]

candidate_users = valid_test_df['user_id'].unique()

relevant_test_df = valid_test_df[valid_test_df['Rating'] >= 3.5]
relevant_dict = relevant_test_df.groupby('user_id')['item_id'].apply(set).to_dict()

print("Building seen_dict using memory-efficient NumPy routing...")
# Filter down to candidate users to save memory
mask = np.isin(train_u_array, candidate_users)
filtered_u = train_u_array[mask]
filtered_i = train_i_array[mask]

# Free the massive full-size arrays immediately
del train_u_array, train_i_array
gc.collect()

import collections
seen_dict = collections.defaultdict(list)
for u, i in zip(filtered_u, filtered_i):
    seen_dict[u].append(i)

seen_dict = dict(seen_dict)  # freeze to standard dict
del filtered_u, filtered_i, valid_test_df, test_df
gc.collect()

def evaluate_map_for_users(target_users, group_name):
    if len(target_users) == 0:
        print(f"No {group_name} in test sample.")
        return

    print(f"\nProcessing MAP@10 for {len(target_users):,} {group_name}...")

    ap_list = []
    batch_size = 2000
    num_batches = int(np.ceil(len(target_users) / batch_size))

    with torch.no_grad():
        all_items_t = torch.arange(num_items, device=device)
        i_e1   = model1.item_emb(all_items_t)
        i_rep1 = model1.item_mlp(i_e1)
        i_b1   = model1.item_bias(all_items_t).squeeze(-1)

    for b in range(num_batches):
        batch_users = target_users[b * batch_size : (b + 1) * batch_size]
        print(f"  Batch {b+1}/{num_batches} ({len(batch_users)} users)...")

        with torch.no_grad():
            users_tensor = torch.tensor(batch_users, dtype=torch.long, device=device)
            u_e1         = model1.user_emb(users_tensor)
            u_rep_all1   = model1.user_mlp(u_e1)
            u_b1         = model1.user_bias(users_tensor).squeeze(-1)

            dense_scores1 = torch.matmul(u_rep_all1, i_rep1.T) + u_b1.unsqueeze(1) + i_b1.unsqueeze(0)

            for idx, u_id in enumerate(batch_users):
                seen = seen_dict.get(u_id, [])
                if seen:
                    dense_scores1[idx, seen] = -float('inf')

            _, top2000_indices1 = torch.topk(dense_scores1, 2000, dim=1)
            top2000_items_all1  = top2000_indices1.cpu().numpy()

        eval_indices = [i for i, u in enumerate(batch_users) if relevant_dict.get(u, set())]
        if not eval_indices:
            continue

        eval_users = [batch_users[i] for i in eval_indices]
        N_eval = len(eval_users)

        eval_top2000 = top2000_items_all1[eval_indices]

        eval_last_items = [last_watched_map.get(u, eval_top2000[i][0]) for i, u in enumerate(eval_users)]
        eval_last_items_tensor = torch.tensor(eval_last_items, device=device)

        with torch.no_grad():
            last_reps = model1.item_mlp(model1.item_emb(eval_last_items_tensor))
            cand_tensors = torch.tensor(eval_top2000, device=device)
            cand_reps = model1.item_mlp(model1.item_emb(cand_tensors))

            cand_norms = F.normalize(cand_reps, p=2, dim=2)
            last_norms = F.normalize(last_reps, p=2, dim=1).unsqueeze(1)
            recency_scores = (cand_norms * last_norms).sum(dim=2).cpu().numpy()

            m1_scores = np.zeros((N_eval, 2000), dtype=np.float32)
            for i, o_idx in enumerate(eval_indices):
                m1_scores[i] = dense_scores1[o_idx, eval_top2000[i]].cpu().numpy()

            u_vecs = u_rep_all1[eval_indices].cpu().numpy()
            cand_vecs = cand_reps.cpu().numpy()

        flat_cands = eval_top2000.flatten()
        pops = np.array([item_pop_map.get(c, 0) for c in flat_cands], dtype=np.float32).reshape(N_eval, 2000)
        avg_rs = np.array([item_avg_map.get(c, 3.5) for c in flat_cands], dtype=np.float32).reshape(N_eval, 2000)
        rating_stds = np.array([item_std_map.get(c, 0.0) for c in flat_cands], dtype=np.float32).reshape(N_eval, 2000)
        years = np.array([movie_years.get(c, 2000) for c in flat_cands], dtype=np.float32).reshape(N_eval, 2000)

        max_pops = pops.max(axis=1, keepdims=True)
        max_pops[max_pops == 0] = 1.0
        rel_pops = pops / max_pops

        user_avgs = np.array([user_avg_map.get(u, 3.5) for u in eval_users], dtype=np.float32).reshape(N_eval, 1)
        rating_gaps = avg_rs - user_avgs

        m1_ranks = np.tile(np.arange(1, 2001, dtype=np.float32) / 2000.0, (N_eval, 1))
        log_pops = np.log1p(pops)

        feat_stack = np.stack([
            m1_scores, recency_scores, log_pops, avg_rs, rating_stds,
            np.broadcast_to(user_avgs, (N_eval, 2000)), m1_ranks, years, rating_gaps, rel_pops
        ], axis=-1).astype(np.float16)

        u_vecs_b = np.broadcast_to(u_vecs[:, np.newaxis, :], (N_eval, 2000, 32)).astype(np.float16)
        cand_vecs = cand_vecs.astype(np.float16)

        lgbm_features = np.concatenate([feat_stack, u_vecs_b, cand_vecs], axis=-1).reshape(-1, 74)

        lgbm_preds = ranker.predict(lgbm_features).reshape(N_eval, 2000)

        for i, u_id in enumerate(eval_users):
            rel_set = relevant_dict[u_id]
            cands = eval_top2000[i]
            preds = lgbm_preds[i]

            order = np.argsort(-preds)
            top10 = cands[order[:10]]

            num_hits = 0.0
            sum_precisions = 0.0
            for rank, rec_item_id in enumerate(top10, 1):
                if rec_item_id in rel_set:
                    num_hits += 1.0
                    sum_precisions += (num_hits / rank)

            ap = sum_precisions / min(len(rel_set), 10)
            ap_list.append(ap)

        del lgbm_features, feat_stack, u_vecs_b, cand_vecs, recency_scores, m1_scores, lgbm_preds
        gc.collect()

    map_at_10 = np.mean(ap_list) if ap_list else 0.0
    print(f"-> {group_name} MAP@10: {map_at_10:.4f}")

cold_test_users = [u for u in candidate_users if u in cold_users_set]
warm_test_users = [u for u in candidate_users if u in warm_users_set]

evaluate_map_for_users(cold_test_users, "Cold Users (<=50 ratings)")
evaluate_map_for_users(warm_test_users, "Warm Users (>50 ratings)")

print("\nEvaluation Complete.")


Using device: cuda

--- Loading Data and Models ---
Loading Model 1 from .\model1_retrieval_2000.pkl...
Loading Model 2 from .\ranker_lightgbm_2000.pkl...
Loading metadata...

--- Calculating Cold/Warm definitions from Training Set ---
Total Cold Users (<= 50 ratings): 27,953
Total Warm Users (> 50 ratings):  259,963
Precomputing statistics for LightGBM features...

--- Evaluating Model 1 (MSE) RMSE ---
Overall RMSE:     0.9833
Cold Users RMSE:  1.0282 (Count: 2,017,244)
Warm Users RMSE:  0.8862 (Count: 9,439,337)

--- Evaluating Model 2 (LightGBM) MAP@10 ---
Building seen_dict using memory-efficient NumPy routing...

Processing MAP@10 for 27,068 Cold Users (<=50 ratings)...
  Batch 1/14 (2000 users)...
  Batch 2/14 (2000 users)...
  Batch 3/14 (2000 users)...
  Batch 4/14 (2000 users)...
  Batch 5/14 (2000 users)...
  Batch 6/14 (2000 users)...
  Batch 7/14 (2000 users)...
  Batch 8/14 (2000 users)...
  Batch 9/14 (2000 users)...
  Batch 10/14 (2000 users)...
  Batch 11/14 (2000 users

# New Section

In [2]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm

# ---------------------------------------------------------
# 1. DEFINE MODEL ARCHITECTURE (Required for unpickling)
# ---------------------------------------------------------
class TwoTowerMSE(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, hidden_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([3.5]))

    def forward(self, user, item):
        u_rep = self.user_mlp(self.user_emb(user))
        i_rep = self.item_mlp(self.item_emb(item))
        dot = (u_rep * i_rep).sum(dim=1)
        u_b = self.user_bias(user).squeeze(-1)
        i_b = self.item_bias(item).squeeze(-1)
        return dot + u_b + i_b + self.global_bias

# ---------------------------------------------------------
# 2. INFERENCE SCRIPT
# ---------------------------------------------------------
def generate_qualifying_predictions():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    data_dir = "/content/drive/Shareddrives/netflixrecommendationsystem/dataset"
    maps_path = os.path.join(data_dir, "maps.pkl")
    model1_path = os.path.join(data_dir, "model1_retrieval_2000.pkl")
    qual_path = os.path.join(data_dir, "qualifying.csv")
    # Modify the output path to save directly to Google Drive
    output_csv_path = os.path.join(data_dir, "predictions_two_tower_model.csv")

    print("Loading ID maps...")
    with open(maps_path, 'rb') as f:
        maps = pickle.load(f)
    user_map = maps['user_map']
    movie_map = maps['movie_map']

    print("Loading Two-Tower Model...")
    model = torch.load(model1_path, map_location=device, weights_only=False)
    model.eval()

    print(f"Reading {qual_path}...")
    df = pd.read_csv(qual_path)

    print("Mapping Raw IDs...")
    # Find mapped IDs, returning NaN if the user/item doesn't exist in training (Cold)
    mapped_u = df['CustomerID'].map(user_map)
    mapped_i = df['MovieID'].map(movie_map)

    # Track which users are completely cold (new users)
    cold_users_mask = mapped_u.isna().values

    # Fill NaN with 0 temporarily so nn.Embedding doesn't crash with IndexError
    user_tensors = torch.tensor(mapped_u.fillna(0).values, dtype=torch.long).to(device)
    item_tensors = torch.tensor(mapped_i.fillna(0).values, dtype=torch.long).to(device)

    print(f"Predicting ratings for {len(df):,} rows...")
    batch_size = 50000
    all_preds = []

    with torch.no_grad():
        for i in tqdm(range(0, len(df), batch_size)):
            u_batch = user_tensors[i:i+batch_size]
            i_batch = item_tensors[i:i+batch_size]

            # Predict full forward pass
            preds = model(u_batch, i_batch)

            # Overwrite Cold Users
            # For users we've never seen, fallback safely to (item_bias + global_mean)
            u_cold_batch = cold_users_mask[i:i+batch_size]
            if u_cold_batch.any():
                item_biases = model.item_bias(i_batch).squeeze(-1)
                fallback_preds = item_biases + model.global_bias
                preds[u_cold_batch] = fallback_preds[u_cold_batch]

            all_preds.extend(preds.cpu().numpy().tolist())

    # Clip ratings to valid Netflix bounds
    all_preds = np.clip(all_preds, 1.0, 5.0)

    print(f"Saving to {output_csv_path}...")
    output_df = pd.DataFrame({
        'MovieID': df['MovieID'].values,
        'CustomerID': df['CustomerID'].values,
        'Prediction': all_preds
    })

    output_df.to_csv(output_csv_path, index=False)
    print("Successfully generated predictions_two_tower_model.csv!")

if __name__ == "__main__":
    generate_qualifying_predictions()


Using device: cpu
Loading ID maps...
Loading Two-Tower Model...
Reading /content/drive/Shareddrives/netflixrecommendationsystem/dataset/qualifying.csv...
Mapping Raw IDs...
Predicting ratings for 2,817,131 rows...


100%|██████████| 57/57 [01:00<00:00,  1.06s/it]


Saving to /content/drive/Shareddrives/netflixrecommendationsystem/dataset/predictions_two_tower_model.csv...
Successfully generated predictions_two_tower_model.csv!


In [5]:
import shutil

drive_path = '/content/drive/Shareddrives/netflixrecommendationsystem/dataset/predictions_two_tower_model.csv'
shutil.move('predictions_two_tower_model.csv', drive_path)
print(f"File 'predictions_two_tower_model.csv' moved to '{drive_path}'")

FileNotFoundError: [Errno 2] No such file or directory: 'predictions_two_tower_model.csv'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
